# 🌙 MoonLight Fusion v3 — o'zbekcha kod yozuvchi LLM, Kaggle T4 ×2, 30 soat

**Bitta `.ipynb`**: data yig'ish → tokenizer → model → 2-GPU DDP trening (pretrain + SFT) → sandbox ichida **pass@1** baholash → HF (Qwen3) formatiga eksport. `moonlight_core.py` va `train_worker.py` notebookning o'zidan `%%writefile` bilan yaratiladi — qo'shimcha fayl yuklash shart emas.

### v3 da nima o'zgardi (eski versiyaga nisbatan)
| Muammo (eski) | Yechim (v3) |
|---|---|
| fp16 da gradient overflow bo'lsa trening **to'xtab qolardi** (GradScaler'ning normal holati xato deb hisoblanardi) | Overflow qadami avtomatik o'tkazib yuboriladi; faqat 30 ta ketma-ket NaN bo'lsa to'xtaydi |
| `BATCH_PER_GPU=1`, 8k token/qadam, har micro-qadamda GPU sinxronizatsiya → T4 juda sekin ishlardi | 131k token/qadam, sinxronizatsiyasiz akkumulyatsiya, OOM bo'lsa micro-batch avtomatik kamayadi |
| Data ~60M token (30 soatda 40+ epoch → overfitting) | ~1B+ token: Python kod + o'zbekcha matn, domen aralash nisbati (`CODE_MIX`) |
| OpenCodeInstruct'da **32% takror masalalar**, val split masala bo'yicha emas (leakage) | `problem_group_id` bo'yicha cheklash va split; 200 ta masala treningdan butunlay ajratilib pass@1 uchun saqlanadi |
| Uzbek datasetda `input` maydonida system-prompt bor, `text` esa "User:/Assistant:" formatida | `messages` dan to'g'ri chat yig'iladi, yagona chat shabloni (maxsus tokenlar) |
| Pure-Python tokenizer (GB lab matnni soatlab kodlardi) | Rust byte-level BPE (32k), o'zbekcha apostrofli so'zlar (o'zbek, g'alaba) bitta bo'lak |
| SEQ=512 → kod javoblarining ko'pi SFT dan tushib qolardi | SEQ=1024 + SFT packing |
| Resume'da bir xil tasodifiy namunalar qayta ko'rilardi | RNG qadam bo'yicha davom etadi |
| Faqat sintaksis tekshiruvi | HumanEval / MBPP / ajratilgan masalalarda **haqiqiy testlar** bilan pass@1 (sandbox) |

### Halol kutilmalar
30 GPU-soat T4×2 da noldan o'qitiladigan ~100M parametrli model **oddiy Python funksiyalarini** yoza oladi, lekin katta modellar (GPT-4, Claude, Qwen-Coder) darajasiga yetmaydi — bu hisoblash hajmi bilan cheklangan. Natijani "his"ga emas, pastdagi **pass@1** raqamlariga qarab baholang. Eng kuchli o'zbekcha kod yordamchisi kerak bo'lsa, tayyor kod modelini (masalan Qwen2.5-Coder) LoRA bilan fine-tune qilish ancha kuchliroq natija beradi.

### Seanslar rejasi (Kaggle GPU seansi ≤ 12 soat, haftalik kvota ≈ 30 soat)
1. **(Tavsiya)** `RUN_MODE='prepare_only'` bilan **CPU** seansida data + tokenizer tayyorlang (GPU kvotasi sarflanmaydi) → `Save Version → Save & Run All`.
2. GPU T4×2 seansi: o'sha Output'ni **Add Input** qiling, `RUN_MODE='all'` → `Save & Run All`. Har seans ~10 soat trening, checkpoint avtomatik.
3. Keyingi seanslarda oxirgi Output'ni Input qiling — notebook eng ko'p rivojlangan checkpointni o'zi topadi. Jami: 26 soat pretrain + 4 soat SFT.

In [ ]:
# 0. Kutubxonalar va GPU tekshiruvi
import os, sys, re, json, math, random, time, hashlib, shutil, gc, ast, subprocess, importlib, importlib.util, warnings
from pathlib import Path
from collections import Counter, defaultdict
from itertools import islice
import numpy as np
import torch

NOTEBOOK_T0 = time.time()   # seans vaqt byudjeti shu paytdan hisoblanadi
warnings.filterwarnings('ignore', category=FutureWarning)
print('Python', sys.version.split()[0], '| PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB, sm{p.major}{p.minor}')
for pkg in ('tokenizers', 'datasets', 'pyarrow'):
    if importlib.util.find_spec(pkg) is None:
        print(f'⚠️ {pkg} topilmadi -> o\'rnatilmoqda');
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
import tokenizers, datasets
print('tokenizers', tokenizers.__version__, '| datasets', datasets.__version__)

## 1. Sozlamalar — faqat shu katakni o'zgartiring

- `RUN_MODE='prepare_only'` — faqat data+tokenizer (CPU yetarli). `'all'` — tayyorlash (kerak bo'lsa) + trening + eval. `'eval_only'` — saqlangan modelni baholash.
- `PROFILE='smoke'` — kichik data va tiny model bilan 2–5 daqiqalik to'liq sinov (pipeline ishlashini tekshirish uchun). Uzoq treningdan oldin bir marta ishlatib ko'ring.
- **`PRESET`, `SEQ`, `VOCAB`, `EXPERTS`, `MTP_WEIGHT` seanslar orasida o'zgarmasligi kerak** — aks holda checkpoint mos kelmaydi (notebook buni tekshiradi).
- `MTP_WEIGHT=0` default: multi-token prediction 1B dan kichik modellarda kod benchmarklarida odatda yomonlashtiradi (Gloeckle va boshq., 2024). Tajriba uchun 0.1–0.3 qo'yish mumkin.
- `SESSION_HOURS` — notebook boshlanganidan beri umumiy vaqt (data tayyorlash ham kiradi). Trening `FINAL_RESERVE_MIN` oldin to'xtab, eval va eksportga vaqt qoldiradi.

In [ ]:
# =========== FAQAT SHU KATAKNI O'ZGARTIRING ===========
PROFILE = 'real'            # 'real' | 'smoke'
RUN_MODE = 'all'            # 'all' | 'prepare_only' | 'eval_only'
SOURCE_MODE = 'auto'        # 'auto' (HF + lokal) | 'local' | 'hf'
LOCAL_PATHS = []            # bo'sh bo'lsa /kaggle/input ichidan avtomatik (jsonl/json/csv/txt/parquet)
MAX_LOCAL_ROWS_PER_FILE = 500_000

# --- HF manbalari (MAX_CHARS ~ belgilar soni; 1e9 ≈ 1 GB matn ≈ 250-300M token) ---
USE_OCI_CODE = True          # JetBrains-Research/OpenCodeInstruct-Clean: 388k Python vazifa+kod+10 test (Apache-2.0)
OCI_PER_PROBLEM_CAP = 8      # bitta masala guruhidan pretrainga maksimal nusxa
OCI_SFT_PER_PROBLEM = 2      # bitta masala guruhidan SFT ga maksimal misol
OCI_EVAL_FRAC = 0.0012       # masala guruhlarining ~0.12% (~200 masala) treningdan butunlay ajratiladi
OCI_VAL_FRAC = 0.01
OCI_EVAL_PROBLEMS = 200
USE_UZ_LM = True             # ulugbek2013/uzbek-lang-lm: 110k o'zbekcha chat/instruction (Apache-2.0)
USE_UZ_WIKI = True;      UZ_WIKI_MAX_CHARS = 250e6      # wikimedia/wikipedia 20231101.uz (CC-BY-SA)
USE_FINEWEB2_UZ = True;  FW2_UZ_MAX_CHARS = 800e6       # HuggingFaceFW/fineweb-2 uzn_Latn (ODC-By)
USE_OPC_ALGO = True;     OPC_ALGO_MAX_CHARS = 400e6     # OpenCoder-LLM/opc-annealing-corpus algorithmic (Python, ODC-By)
USE_OPC_SNIPPET = True;  OPC_SNIPPET_MAX_CHARS = 300e6  # OpenCoder-LLM/opc-annealing-corpus synthetic_code_snippet (Python)
USE_CODEPARROT = True;   CODEPARROT_MAX_CHARS = 1.5e9   # codeparrot/codeparrot-clean: faqat MIT/Apache/BSD/ISC/CC0 fayllar
DECONTAMINATE = True         # HumanEval/MBPP bilan 10-gram ustma-ust tushgan kod hujjatlarini chiqarib tashlash
VAL_PCT = 0.5                # guruhsiz manbalar uchun val ulushi (%)
VAL_MAX_CHARS_PER_SOURCE = 4e6
CURATED_REPEAT = 3           # qo'lda yozilgan o'zbekcha kod misollari SFT da necha marta takrorlanadi

# --- Tokenizer ---
VOCAB = 32768
TOKENIZER_SAMPLE_CHARS = 200e6

# --- Model ---
PRESET = 'base'              # tiny | small (~49M) | base (~104M, tavsiya) | large (~176M)
SEQ = 1024
EXPERTS = 0; TOP_K_EXPERTS = 2   # MoE eksperimental, T4 da sekin
MTP_WEIGHT = 0.0

# --- Trening ---
BATCH_PER_GPU = 8; ACCUM = 8     # 2 GPU x 8 x 8 x 1024 = 131k token/qadam (OOM bo'lsa avtomatik moslashadi)
ACCUM_SFT = 4
LR_PRETRAIN = 1e-3; LR_SFT = 2e-4
WARMUP_STEPS = 500; WARMUP_STEPS_SFT = 100
MIN_LR_RATIO = 0.1; DECAY_FRAC = 0.2   # WSD: pretrain vaqtining oxirgi 20% ida LR pasayadi
WEIGHT_DECAY = 0.1; GRAD_CLIP = 1.0
CODE_MIX = 0.7               # pretrain: 70% kod, 30% o'zbekcha matn
CODE_SFT_RATIO = 0.7         # SFT: 70% kod misollari
PRETRAIN_HOURS = 26.0; SFT_HOURS = 4.0
SESSION_HOURS = 11.25        # Kaggle 12 soatlik limitidan xavfsiz
FINAL_RESERVE_MIN = 40       # trening tugagach eval + eksport uchun
SAVE_EVERY_MIN = 20; EVAL_EVERY_MIN = 30; EVAL_BATCHES = 24; LOG_EVERY_STEPS = 20
EARLY_STOP_PATIENCE = 0      # pretrain uchun (0 = o'chiq)
EARLY_STOP_PATIENCE_SFT = 4  # SFT val loss 4 marta yaxshilanmasa to'xtaydi (overfittingdan himoya)
MAX_STEPS_PER_SESSION = 0
COMPILE = False              # torch.compile (~1.2-1.4x tezroq bo'lishi mumkin, lekin riskliroq)
GRAD_CKPT = False            # xotira yetmasa True (sekinroq)
FORCE_CPU = False; NPROC = 'auto'
SEED = 42

# --- Baholash / eksport ---
RUN_EVAL = True
RUN_EXEC_EVAL = True         # model kodini sandbox jarayonda testlar bilan ishga tushirish (pass@1)
EVAL_OCI_N = 200; EVAL_HUMANEVAL = True; EVAL_MBPP = True; EVAL_BASE_HUMANEVAL = True
EVAL_MAX_NEW = 384; EVAL_BATCH = 16
EXPORT_HF = True             # export/hf_qwen3: transformers.AutoModelForCausalLM bilan ochiladi

# Oldingi seans Output'i: bo'sh qolsa /kaggle/input ichidan avtomatik topiladi
RESUME_INPUT = ''
OUTPUT_ROOT = '/kaggle/working/moonlight_30h' if Path('/kaggle/working').is_dir() else str(Path.cwd() / 'moonlight_30h')
RAW_DIR = '/tmp/moonlight_raw'   # vaqtinchalik xom data (Output'ga saqlanmaydi)
KEEP_RAW = False

SMOKE_ROWS = None
if PROFILE == 'smoke':
    PRESET = 'tiny'; VOCAB = 1024; SEQ = 128; BATCH_PER_GPU = 4; ACCUM = 2; ACCUM_SFT = 1
    PRETRAIN_HOURS = 0.01; SFT_HOURS = 0.008; WARMUP_STEPS = 3; WARMUP_STEPS_SFT = 2
    SMOKE_ROWS = 400; OCI_EVAL_FRAC = 0.05; OCI_VAL_FRAC = 0.1; OCI_EVAL_PROBLEMS = 8
    UZ_WIKI_MAX_CHARS = FW2_UZ_MAX_CHARS = OPC_ALGO_MAX_CHARS = OPC_SNIPPET_MAX_CHARS = CODEPARROT_MAX_CHARS = 3e5
    VAL_PCT = 10; TOKENIZER_SAMPLE_CHARS = 2e6
    EVAL_BATCHES = 2; SAVE_EVERY_MIN = 1; EVAL_EVERY_MIN = 0.3; LOG_EVERY_STEPS = 1
    EVAL_OCI_N = 8; EVAL_MAX_NEW = 32; EVAL_BATCH = 4

assert PROFILE in ('real', 'smoke') and RUN_MODE in ('all', 'prepare_only', 'eval_only')
assert SOURCE_MODE in ('auto', 'local', 'hf') and PRESET in ('tiny', 'small', 'base', 'large')
assert 300 <= VOCAB <= 65535 and SEQ >= 32 and BATCH_PER_GPU >= 1 and ACCUM >= 1
assert EXPERTS == 0 or 1 <= TOP_K_EXPERTS <= EXPERTS
ROOT = Path(OUTPUT_ROOT); PROC = ROOT / 'processed'; RAW = Path(RAW_DIR)
print('ROOT:', ROOT, '| MODEL:', PRESET, '| SEQ:', SEQ, '| VOCAB:', VOCAB, '| PROFILE:', PROFILE, '| RUN_MODE:', RUN_MODE)

## 2. Resume: oldingi seans natijalarini qaytarish

Kaggle'da `/kaggle/working` keyingi seansga o'tmaydi. Oldingi seans Output'ini **Add Input** qiling — bu katak `/kaggle/input` ichidan `moonlight_30h/processed/meta.json` bor kataloglarni topadi va **eng ko'p o'qitilganini** tanlaydi (hammasini log qiladi). Aniq bittasini tanlash uchun `RESUME_INPUT` ga yo'lni yozing.

In [ ]:
# 2. Resume
ROOT.mkdir(parents=True, exist_ok=True)

def restore_from(src):
    """/kaggle/input read-only: nusxadan keyin yozish huquqini qaytaramiz (checkpoint ustiga yozish uchun)."""
    shutil.copytree(src, ROOT, dirs_exist_ok=True, copy_function=shutil.copyfile)
    for p in [ROOT, *ROOT.rglob('*')]:
        try:
            p.chmod(p.stat().st_mode | 0o700 if p.is_dir() else p.stat().st_mode | 0o600)
        except OSError:
            pass

def _progress(d):
    """(o'qitilgan soatlar, oxirgi yangilanish) — eng rivojlangan checkpointni tanlash uchun."""
    try:
        m = json.loads((d / 'manifest.json').read_text('utf-8'))
        return sum(v.get('elapsed_hours', 0) for v in m.get('stages', {}).values()), m.get('last_updated_utc', '')
    except Exception:
        return 0.0, ''

if not RESUME_INPUT and Path('/kaggle/input').exists():
    cands = sorted({p.parent.parent for p in Path('/kaggle/input').rglob('meta.json')
                    if p.parent.name == 'processed' and p.parent.parent.name == 'moonlight_30h'})
    for c in cands:
        print('  topildi:', c, '| o\'qitilgan soat: %.2f | %s' % _progress(c))
    if cands:
        RESUME_INPUT = str(max(cands, key=_progress))
        print('Avtomatik tanlandi:', RESUME_INPUT)

if RESUME_INPUT:
    src = Path(RESUME_INPUT)
    if not (src / 'processed' / 'meta.json').is_file():
        raise FileNotFoundError(f'{src}/processed/meta.json topilmadi')
    src_id = json.loads((src / 'processed' / 'meta.json').read_text('utf-8'))['prepared_id']
    if (PROC / 'meta.json').exists():
        cur_id = json.loads((PROC / 'meta.json').read_text('utf-8'))['prepared_id']
        if cur_id != src_id:
            raise RuntimeError('ROOT da boshqa dataset bor. Sessiyani qayta ishga tushiring yoki OUTPUT_ROOT ni o\'zgartiring.')
        if _progress(src)[0] > _progress(ROOT)[0]:
            restore_from(src)
        print('ROOT allaqachon tiklangan:', ROOT)
    else:
        t = time.time()
        restore_from(src)
        print(f'✅ Tiklandi: {src} -> {ROOT} ({time.time() - t:.0f}s) | o\'qitilgan soat: {_progress(ROOT)[0]:.2f}')
else:
    print('Yangi trening. Oldingi Output bo\'lsa uni Add Input qiling.')

## 3. Yadro: tokenizer + model + sandbox (`moonlight_core.py`)

GPT uslubidagi decoder; bloklar Llama/Qwen3 kabi: **RoPE, RMSNorm, SwiGLU, GQA, QK-norm, tied embeddings**. Ixtiyoriy MoE va DeepSeek-V3 uslubidagi MTP. Dense rejimdagi vaznlar Qwen3 formatiga 1:1 mos (eksportda logitlar solishtirib tekshiriladi). Fayl `ROOT` ichiga yoziladi va worker ham shuni import qiladi.

In [ ]:
%%writefile {ROOT}/moonlight_core.py
# AUTOGENERATED by the MoonLight notebook (%%writefile). train_worker.py ham shu faylni import qiladi.
"""MoonLight Fusion core: tokenizer, model, generation, sandboxed code evaluation.

Arxitektura: GPT uslubidagi decoder-only + Llama/Qwen3 bloklari
(RoPE, RMSNorm, SwiGLU, GQA, QK-norm, tied embeddings). Ixtiyoriy: sparse MoE va
DeepSeek-V3 uslubidagi MTP moduli. Dense rejimdagi vaznlar Qwen3 formatiga
1:1 eksport qilinadi (transformers / vLLM / llama.cpp bilan ishlatish uchun).
"""
from __future__ import annotations

import math
import os
import re
import signal
import subprocess
import sys
import tempfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, fields

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as _ckpt

try:  # fork'dan OLDIN import qilinadi (preexec_fn ichida import qilish deadlock berishi mumkin)
    import resource as _resource
except ImportError:  # Windows
    _resource = None

# ----------------------------------------------------------------------------
# Tokenizer: byte-level BPE (HF `tokenizers`, Rust). Har qanday UTF-8 matn
# yo'qotishsiz qaytariladi. Pre-tokenizer o'zbekcha apostrofli so'zlarni
# (o'zbek, g'alaba, ma'no) bitta bo'lak sifatida saqlaydi, kod indentatsiyasi
# uchun bo'shliq ketma-ketliklarini ajratadi va raqamlarni 1-3 tadan bo'ladi.
# ----------------------------------------------------------------------------
SPECIAL_TOKENS = ["<|pad|>", "<|bos|>", "<|eos|>", "<|user|>", "<|assistant|>", "<|system|>"]
PRETOK_PATTERN = (
    r"[^\r\n\p{L}\p{N}]?\p{L}+(?:['ʻʼ‘’]\p{L}+)*"
    r"|\p{N}{1,3}"
    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"
    r"|\s*[\r\n]+"
    r"|\s+(?!\S)"
    r"|\s+"
)
_SPECIAL_RE = re.compile("|".join(re.escape(t) for t in SPECIAL_TOKENS))


def strip_special(text: str) -> str:
    """Ma'lumot ichida tasodifan uchragan maxsus token matnlarini olib tashlaydi."""
    return _SPECIAL_RE.sub("", text) if "<|" in text else text


class MoonTokenizer:
    def __init__(self, tk):
        self.tk = tk
        self.vocab_size = tk.get_vocab_size()
        ids = [tk.token_to_id(t) for t in SPECIAL_TOKENS]
        if any(i is None for i in ids):
            raise ValueError("Tokenizer maxsus tokenlari topilmadi")
        (self.pad_id, self.bos_id, self.eos_id,
         self.user_id, self.assistant_id, self.system_id) = ids

    @classmethod
    def train(cls, texts, vocab_size: int, min_frequency: int = 2) -> "MoonTokenizer":
        from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers
        tk = Tokenizer(models.BPE())
        tk.pre_tokenizer = pre_tokenizers.Sequence([
            pre_tokenizers.Split(Regex(PRETOK_PATTERN), behavior="isolated"),
            pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
        ])
        tk.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(
            vocab_size=vocab_size, min_frequency=min_frequency,
            special_tokens=SPECIAL_TOKENS,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
            show_progress=False)
        tk.train_from_iterator(texts, trainer=trainer)
        return cls(tk)

    def encode(self, text: str) -> list[int]:
        return self.tk.encode(text, add_special_tokens=False).ids

    def encode_batch(self, texts: list[str]) -> list[list[int]]:
        return [e.ids for e in self.tk.encode_batch(texts, add_special_tokens=False)]

    def decode(self, ids, skip_special: bool = True) -> str:
        return self.tk.decode([int(i) for i in ids], skip_special_tokens=skip_special)

    def save(self, path):
        self.tk.save(str(path))

    @classmethod
    def load(cls, path) -> "MoonTokenizer":
        from tokenizers import Tokenizer
        return cls(Tokenizer.from_file(str(path)))


def chat_ids(tok: MoonTokenizer, turns, add_generation_prompt: bool = False):
    """Chat shabloni (o'qitish va inference uchun AYNAN bir xil):
    <|bos|><|user|>\\n{savol}\\n<|assistant|>\\n{javob}<|eos|> ...
    Qaytaradi: (ids, loss_mask) — loss faqat yordamchi javobi + <|eos|> da.
    """
    ids, mask = [tok.bos_id], [0]
    for role, text in turns:
        text = (text or "").strip()
        if role == "assistant":
            head = [tok.assistant_id] + tok.encode("\n")
            body = tok.encode(text) + [tok.eos_id]
            ids += head + body
            mask += [0] * len(head) + [1] * len(body)
        else:
            rid = tok.system_id if role == "system" else tok.user_id
            part = [rid] + tok.encode("\n" + text + "\n")
            ids += part
            mask += [0] * len(part)
    if add_generation_prompt:
        head = [tok.assistant_id] + tok.encode("\n")
        ids += head
        mask += [0] * len(head)
    return ids, mask


CHAT_TEMPLATE = (
    "{{ bos_token }}{% for m in messages %}"
    "{% if m['role'] == 'assistant' %}<|assistant|>\n{{ m['content'] | trim }}<|eos|>"
    "{% elif m['role'] == 'system' %}<|system|>\n{{ m['content'] | trim }}\n"
    "{% else %}<|user|>\n{{ m['content'] | trim }}\n{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}<|assistant|>\n{% endif %}"
)

# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------


@dataclass
class Config:
    vocab_size: int = 32768
    dim: int = 576
    layers: int = 24
    heads: int = 9
    kv_heads: int = 3
    hidden: int = 1536
    max_seq: int = 1024
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    experts: int = 0
    top_k: int = 2
    dropout: float = 0.0
    mtp_weight: float = 0.0
    moe_aux_weight: float = 0.01
    grad_ckpt: bool = False

    @property
    def head_dim(self) -> int:
        return self.dim // self.heads

    def validate(self):
        assert self.dim % self.heads == 0, "dim heads ga bo'linishi kerak"
        assert self.heads % self.kv_heads == 0, "heads kv_heads ga bo'linishi kerak"
        assert self.head_dim % 2 == 0, "RoPE uchun head_dim juft bo'lishi kerak"
        assert not self.experts or 1 <= self.top_k <= self.experts
        assert 260 <= self.vocab_size <= 65535, "uint16 token saqlash uchun vocab <= 65535"
        return self

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        names = {f.name for f in fields(cls)}
        return cls(**{k: v for k, v in d.items() if k in names})


# Chuqur-va-ingichka (MobileLLM / SmolLM uslubi) presetlar. Parametrlar vocab=32768 bilan.
PRESETS = {
    "tiny": dict(dim=128, layers=4, heads=4, kv_heads=2, hidden=384),        # smoke test
    "small": dict(dim=512, layers=12, heads=8, kv_heads=2, hidden=1408),     # ~49M
    "base": dict(dim=576, layers=24, heads=9, kv_heads=3, hidden=1536),      # ~104M (T4x2 30h uchun tavsiya)
    "large": dict(dim=768, layers=24, heads=12, kv_heads=4, hidden=2048),    # ~176M (30h ga kam token)
}


class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        z = x.float()
        z = z * torch.rsqrt(z.pow(2).mean(-1, keepdim=True) + self.eps)
        return (z * self.scale.float()).to(x.dtype)


def rope_tables(max_seq: int, head_dim: int, theta: float):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    t = torch.arange(max_seq, dtype=torch.float32)
    freqs = torch.outer(t, inv)
    emb = torch.cat((freqs, freqs), dim=-1)          # Llama/Qwen "rotate_half" konvensiyasi
    return emb.cos(), emb.sin()


def apply_rope(x, cos, sin):
    d = x.size(-1) // 2
    xf = x.float()
    rot = torch.cat((-xf[..., d:], xf[..., :d]), dim=-1)
    return (xf * cos + rot * sin).to(x.dtype)


class Attention(nn.Module):
    def __init__(self, c: Config):
        super().__init__()
        self.h, self.kvh, self.hd = c.heads, c.kv_heads, c.head_dim
        self.q = nn.Linear(c.dim, c.heads * self.hd, bias=False)
        self.k = nn.Linear(c.dim, c.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, c.kv_heads * self.hd, bias=False)
        self.o = nn.Linear(c.heads * self.hd, c.dim, bias=False)
        self.q_norm = RMSNorm(self.hd, c.norm_eps)
        self.k_norm = RMSNorm(self.hd, c.norm_eps)
        self.dropout = c.dropout

    def forward(self, x, cos, sin, past=None, attn_mask=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_norm(self.q(x).view(B, T, self.h, self.hd)).transpose(1, 2)
        k = self.k_norm(self.k(x).view(B, T, self.kvh, self.hd)).transpose(1, 2)
        v = self.v(x).view(B, T, self.kvh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], dim=-2)
            v = torch.cat([past[1], v], dim=-2)
        cache = (k, v) if use_cache else None
        if self.kvh != self.h:
            # T4 (sm75) da enable_gqa math-kernelga tushib qoladi; repeat esa mem-efficient kernelni saqlaydi.
            rep = self.h // self.kvh
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)
        drop = self.dropout if self.training else 0.0
        if attn_mask is not None:
            a = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=drop)
        else:
            if past is not None and T > 1:
                raise ValueError("KV-cache bilan ko'p tokenli qadam uchun attn_mask kerak")
            a = F.scaled_dot_product_attention(q, k, v, dropout_p=drop, is_causal=(past is None and T > 1))
        return self.o(a.transpose(1, 2).reshape(B, T, -1)), cache


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.gate = nn.Linear(dim, hidden, bias=False)
        self.up = nn.Linear(dim, hidden, bias=False)
        self.down = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class SparseMoE(nn.Module):
    """Top-k routing + shared expert + Switch uslubidagi load-balancing loss.
    Eksperimental: Python dispatch T4 da odatda dense'dan SEKINROQ."""

    def __init__(self, c: Config):
        super().__init__()
        h = max(64, c.hidden // 2)
        self.router = nn.Linear(c.dim, c.experts, bias=False)
        self.experts = nn.ModuleList([SwiGLU(c.dim, h) for _ in range(c.experts)])
        self.shared = SwiGLU(c.dim, h)
        self.n, self.k = c.experts, c.top_k

    def forward(self, x):
        flat = x.reshape(-1, x.size(-1))
        probs = self.router(flat).float().softmax(-1)
        weights, idx = probs.topk(self.k, dim=-1)
        weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-9)
        out = self.shared(flat)
        for e, expert in enumerate(self.experts):
            tok, slot = torch.where(idx == e)
            if tok.numel():
                out = out.index_add(0, tok, expert(flat[tok]) * weights[tok, slot, None].to(flat.dtype))
        load = F.one_hot(idx, self.n).float().mean(dim=(0, 1)).detach()
        aux = self.n * (probs.mean(0) * load).sum()
        return out.view_as(x), aux


class Block(nn.Module):
    def __init__(self, c: Config, dense: bool = False):
        super().__init__()
        self.n1 = RMSNorm(c.dim, c.norm_eps)
        self.att = Attention(c)
        self.n2 = RMSNorm(c.dim, c.norm_eps)
        self.moe = bool(c.experts) and not dense
        self.ff = SparseMoE(c) if self.moe else SwiGLU(c.dim, c.hidden)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, cos, sin, past=None, attn_mask=None, use_cache=False):
        a, cache = self.att(self.n1(x), cos, sin, past=past, attn_mask=attn_mask, use_cache=use_cache)
        x = x + self.drop(a)
        if self.moe:
            f, aux = self.ff(self.n2(x))
        else:
            f, aux = self.ff(self.n2(x)), x.new_zeros((), dtype=torch.float32)
        return x + self.drop(f), aux, cache


class MTPModule(nn.Module):
    """DeepSeek-V3 uslubidagi multi-token prediction: h_t va Emb(x_{t+1}) dan x_{t+2} ni bashorat qiladi."""

    def __init__(self, c: Config):
        super().__init__()
        self.norm_h = RMSNorm(c.dim, c.norm_eps)
        self.norm_e = RMSNorm(c.dim, c.norm_eps)
        self.proj = nn.Linear(2 * c.dim, c.dim, bias=False)
        self.block = Block(c, dense=True)
        self.norm = RMSNorm(c.dim, c.norm_eps)

    def forward(self, h, next_emb, cos, sin):
        z = self.proj(torch.cat([self.norm_h(h), self.norm_e(next_emb)], dim=-1))
        z, _, _ = self.block(z, cos, sin)
        return self.norm(z)


def masked_ce(logits, targets):
    """-100 belgilangan tokenlarni hisobga olmaydi; hamma token yashirin bo'lsa NaN emas, 0 qaytaradi."""
    t = targets.reshape(-1)
    s = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), t, ignore_index=-100, reduction="sum")
    return s / (t != -100).sum().clamp_min(1)


class MoonLightFusion(nn.Module):
    def __init__(self, c: Config):
        super().__init__()
        self.cfg = c.validate()
        self.embed = nn.Embedding(c.vocab_size, c.dim)
        self.drop = nn.Dropout(c.dropout)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.layers)])
        self.norm = RMSNorm(c.dim, c.norm_eps)
        self.mtp = MTPModule(c) if c.mtp_weight > 0 else None
        cos, sin = rope_tables(c.max_seq, c.head_dim, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        # GPT-2 uslubi: residual chiqish proyeksiyalarini chuqurlikka qarab kichraytirish.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith("att.o.weight") or name.endswith("ff.down.weight") or name.endswith(".down.weight"):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self, non_embedding: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        return n - self.embed.weight.numel() if non_embedding else n

    def forward(self, ids, targets=None, targets2=None, past=None, attn_mask=None, use_cache=False):
        B, T = ids.shape
        offset = past[0][0].size(-2) if past is not None else 0
        if offset + T > self.cfg.max_seq:
            raise ValueError(f"ketma-ketlik {offset + T} > max_seq={self.cfg.max_seq}")
        cos = self.rope_cos[offset:offset + T]
        sin = self.rope_sin[offset:offset + T]
        x = self.drop(self.embed(ids))
        aux = x.new_zeros((), dtype=torch.float32)
        caches = []
        for i, b in enumerate(self.blocks):
            p = None if past is None else past[i]
            if self.cfg.grad_ckpt and self.training and targets is not None:
                x, a, c = _ckpt(b, x, cos, sin, use_reentrant=False)
            else:
                x, a, c = b(x, cos, sin, past=p, attn_mask=attn_mask, use_cache=use_cache)
            aux = aux + a
            if use_cache:
                caches.append(c)
        h = self.norm(x)
        if targets is None:
            logits = F.linear(h[:, -1:] if use_cache else h, self.embed.weight)
            return logits, None, (caches if use_cache else None)
        loss = masked_ce(F.linear(h, self.embed.weight), targets)
        if self.mtp is not None and targets2 is not None:
            nxt = self.embed(targets.clamp_min(0))
            z = self.mtp(x, nxt, cos, sin)
            loss = loss + self.cfg.mtp_weight * masked_ce(F.linear(z, self.embed.weight), targets2)
        if self.cfg.experts:
            loss = loss + self.cfg.moe_aux_weight * aux / len(self.blocks)
        return None, loss, None  # o'qitishda logits qaytarilmaydi (~0.5 GB xotira tejaladi)

    # ---------------------------------------------------------------- generation
    @torch.inference_mode()
    def generate_batch(self, prompts, max_new_tokens=256, temperature=0.0, top_k=0, top_p=1.0,
                       stop_ids=(2,), pad_id=0, device=None):
        """Left-padding + KV-cache bilan batch generatsiya. Qaytaradi: har bir prompt uchun yangi tokenlar."""
        was_training = self.training
        self.eval()
        device = device or next(self.parameters()).device
        B = len(prompts)
        L = max(len(p) for p in prompts)
        max_new = max(0, min(max_new_tokens, self.cfg.max_seq - L))
        ids = torch.full((B, L), pad_id, dtype=torch.long)
        valid = torch.zeros((B, L), dtype=torch.bool)
        for i, p in enumerate(prompts):
            ids[i, L - len(p):] = torch.tensor(p, dtype=torch.long)
            valid[i, L - len(p):] = True
        ids, valid = ids.to(device), valid.to(device)
        causal = torch.ones(L, L, dtype=torch.bool, device=device).tril()
        mask = causal[None, None] & valid[:, None, None, :]
        mask = mask | torch.eye(L, dtype=torch.bool, device=device)[None, None]  # pad qatorlari NaN bermasligi uchun
        logits, _, cache = self(ids, attn_mask=mask, use_cache=True)
        stop = torch.tensor(list(stop_ids), device=device)
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        for _ in range(max_new):
            nxt = sample_next(logits[:, -1].float(), temperature, top_k, top_p)
            nxt = torch.where(done, torch.full_like(nxt, pad_id), nxt)
            out.append(nxt)
            done = done | torch.isin(nxt, stop)
            if bool(done.all()):
                break
            valid = torch.cat([valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, _, cache = self(nxt[:, None], past=cache, attn_mask=valid[:, None, None, :], use_cache=True)
        self.train(was_training)
        if not out:
            return [[] for _ in range(B)]
        gen = torch.stack(out, dim=1).tolist()
        stops = set(int(s) for s in stop_ids)
        res = []
        for row in gen:
            cut = []
            for t in row:
                if t in stops:
                    break
                cut.append(t)
            res.append(cut)
        return res

    def generate(self, ids, max_new_tokens=256, temperature=0.0, top_k=0, top_p=1.0, stop_ids=(2,), pad_id=0, device=None):
        return self.generate_batch([list(ids)], max_new_tokens, temperature, top_k, top_p, stop_ids, pad_id, device)[0]


def sample_next(logits, temperature=0.0, top_k=0, top_p=1.0):
    if temperature <= 0:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k and top_k < logits.size(-1):
        kth = torch.topk(logits, top_k, dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p
        sl = sl.masked_fill(drop, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(-1)


def load_checkpoint_model(path, device="cpu", dtype=None):
    """O'zingiz yaratgan checkpointni yuklash (weights_only=False — begona .pt fayllarni ochmang!)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model = MoonLightFusion(Config.from_dict(ck["config"]))
    model.load_state_dict(ck["model"])
    model = model.to(device)
    if dtype is not None:
        model = model.to(dtype)
        for b in ("rope_cos", "rope_sin"):  # RoPE jadvallari fp32 da qolishi kerak
            model._buffers[b] = model._buffers[b].float()
    return model.eval(), ck

# ----------------------------------------------------------------------------
# Qwen3 (HF transformers) formatiga eksport — faqat dense (EXPERTS=0) rejim.
# ----------------------------------------------------------------------------


def qwen3_config_dict(c: Config, tok: MoonTokenizer) -> dict:
    return {
        "architectures": ["Qwen3ForCausalLM"], "model_type": "qwen3",
        "vocab_size": c.vocab_size, "hidden_size": c.dim, "intermediate_size": c.hidden,
        "num_hidden_layers": c.layers, "num_attention_heads": c.heads,
        "num_key_value_heads": c.kv_heads, "head_dim": c.head_dim,
        "hidden_act": "silu", "max_position_embeddings": c.max_seq,
        "rms_norm_eps": c.norm_eps, "rope_theta": c.rope_theta, "rope_scaling": None,
        "attention_bias": False, "attention_dropout": 0.0, "tie_word_embeddings": True,
        "use_sliding_window": False, "sliding_window": None, "max_window_layers": c.layers,
        "initializer_range": 0.02, "use_cache": True, "torch_dtype": "float16",
        "bos_token_id": tok.bos_id, "eos_token_id": tok.eos_id, "pad_token_id": tok.pad_id,
    }


def to_qwen3_state_dict(model: MoonLightFusion) -> dict:
    if model.cfg.experts:
        raise ValueError("MoE modelni Qwen3 formatiga eksport qilib bo'lmaydi")
    sd = {"model.embed_tokens.weight": model.embed.weight, "model.norm.weight": model.norm.scale}
    for i, b in enumerate(model.blocks):
        p = f"model.layers.{i}."
        sd.update({
            p + "input_layernorm.weight": b.n1.scale,
            p + "post_attention_layernorm.weight": b.n2.scale,
            p + "self_attn.q_proj.weight": b.att.q.weight,
            p + "self_attn.k_proj.weight": b.att.k.weight,
            p + "self_attn.v_proj.weight": b.att.v.weight,
            p + "self_attn.o_proj.weight": b.att.o.weight,
            p + "self_attn.q_norm.weight": b.att.q_norm.scale,
            p + "self_attn.k_norm.weight": b.att.k_norm.scale,
            p + "mlp.gate_proj.weight": b.ff.gate.weight,
            p + "mlp.up_proj.weight": b.ff.up.weight,
            p + "mlp.down_proj.weight": b.ff.down.weight,
        })
    return {k: v.detach().to("cpu", torch.float16).contiguous() for k, v in sd.items()}

# ----------------------------------------------------------------------------
# Sandbox: model yozgan kodni testlar bilan ALOHIDA jarayonda, vaqt/xotira
# cheklovi bilan, xavfli funksiyalar o'chirilgan holda ishga tushirish
# (OpenAI human-eval `reliability_guard` uslubi). Bu to'liq xavfsizlik
# kafolati emas; faqat bir martalik Kaggle konteynerida ishlating.
# ----------------------------------------------------------------------------
GUARD_SRC = r'''
import builtins as _b, io as _io, os as _os, shutil as _sh, subprocess as _sp, sys as _sys, faulthandler as _fh
_fh.disable()
_cwd = _os.path.realpath(_os.getcwd())
_real_open = _b.open
def _safe_open(file, mode="r", *a, **k):
    if isinstance(file, int) or any(ch in mode for ch in "wax+"):
        p = None if isinstance(file, int) else _os.path.realpath(_os.fspath(file))
        if p is None or not p.startswith(_cwd):
            raise PermissionError("sandbox: faqat vaqtinchalik katalogga yozish mumkin")
    return _real_open(file, mode, *a, **k)
_b.open = _safe_open; _io.open = _safe_open
for _n in ("kill", "system", "putenv", "remove", "removedirs", "rmdir", "fchdir", "setuid", "fork",
           "forkpty", "killpg", "rename", "renames", "truncate", "replace", "unlink", "fchmod",
           "fchown", "chmod", "chown", "chroot", "lchown", "chdir", "execv", "execve", "execvp",
           "spawnv", "spawnve", "popen"):
    if hasattr(_os, _n):
        setattr(_os, _n, None)
_sh.rmtree = None; _sh.move = None; _sh.chown = None
_sp.Popen = None; _sp.run = None; _sp.call = None; _sp.check_output = None
_b.exit = None; _b.quit = None
for _m in ("ipdb", "joblib", "resource", "psutil", "tkinter"):
    _sys.modules[_m] = None
del _n, _m
'''


def _limits(mem_mb: int, cpu_s: int):
    mem, r = mem_mb << 20, _resource

    def fn():
        try:
            r.setrlimit(r.RLIMIT_AS, (mem, mem))
            r.setrlimit(r.RLIMIT_CPU, (cpu_s, cpu_s + 1))
            r.setrlimit(r.RLIMIT_FSIZE, (16 << 20, 16 << 20))
        except Exception:
            pass
    return fn


def run_program(program: str, timeout: float = 6.0, mem_mb: int = 1024):
    """(passed, xato_matni) qaytaradi. Kod alohida python -I jarayonida bajariladi."""
    with tempfile.TemporaryDirectory(prefix="moon_exec_") as d:
        path = os.path.join(d, "prog.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(GUARD_SRC + "\n" + program + "\n")
        env = {"PATH": os.environ.get("PATH", "/usr/bin:/bin"), "PYTHONHASHSEED": "0",
               "OMP_NUM_THREADS": "1", "HOME": d, "PYTHONDONTWRITEBYTECODE": "1"}
        try:
            proc = subprocess.Popen([sys.executable, "-I", path], cwd=d, env=env,
                                    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
                                    start_new_session=True,
                                    preexec_fn=_limits(mem_mb, int(timeout) + 1) if _resource is not None else None)
        except Exception as e:  # noqa: BLE001
            return False, f"launch: {e!r}"
        try:
            _, err = proc.communicate(timeout=timeout)
            return proc.returncode == 0, err.decode("utf-8", "replace")[-400:]
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except Exception:
                proc.kill()
            proc.communicate()
            return False, "timeout"


def run_many(programs, workers: int = 4, timeout: float = 6.0):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(lambda p: run_program(p, timeout), programs))


_FENCE = re.compile(r"```(?:python|py|Python)?\s*\n(.*?)(?:```|$)", re.S)


def extract_code(text: str) -> str:
    """Markdown ```python blokini (bo'lsa) yoki butun javobni qaytaradi."""
    m = _FENCE.search(text)
    code = m.group(1) if m else text
    return code.strip("\n")


HE_STOPS = ("\ndef ", "\nclass ", "\nif __name__", "\nprint(", "\n#", "\nassert ", '\n"""', "\n```")


def truncate_completion(text: str, stops=HE_STOPS) -> str:
    cut = len(text)
    for s in stops:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut]

In [ ]:
# 3.1 Yadroni import qilish va presetlar hajmi
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import moonlight_core as mc
importlib.reload(mc)

with torch.device('meta'):
    for name, spec in mc.PRESETS.items():
        m_ = mc.MoonLightFusion(mc.Config(vocab_size=VOCAB, max_seq=SEQ, **spec))
        print(f"{name:6s}: {m_.num_params() / 1e6:6.1f}M parametr (embedding'siz {m_.num_params(True) / 1e6:6.1f}M)"
              + ('   <-- tanlangan' if name == PRESET else ''))
del m_

## 4. Dataset: manbalar, tozalash, dedup, decontamination

**HF manbalari tekshirildi (2026-11):**

| Manba | Nima | Topilgan muammo → yechim |
|---|---|---|
| [JetBrains-Research/OpenCodeInstruct-Clean](https://huggingface.co/datasets/JetBrains-Research/OpenCodeInstruct-Clean) | 388,629 Python vazifa (`instruction`, `code`, 10 ta `testcase`), barcha testlardan o'tgan | 27.7% funksional takror + semantik oilalar (bitta guruhda 500+ nusxa) → `problem_group_id` bo'yicha cheklash. Ko'rsatmalar inglizcha → o'zbekcha shablonlar bilan o'raladi (tarjima emas). Parquet tartiblangan → endi butun dataset o'qiladi (eski: birinchi 150k qator) |
| [ulugbek2013/uzbek-lang-lm](https://huggingface.co/datasets/ulugbek2013/uzbek-lang-lm) | 110k train + 8.6k val o'zbekcha chat/instruction, 14 domen (shu jumladan dasturlash) | `input` maydonida ko'pincha system-prompt; `text` "System:/User:/Assistant:" formatida → `messages` dan chat tuziladi, system-promptlar tashlanadi; kodli javoblar kod-SFT deb belgilanadi |
| [wikimedia/wikipedia `20231101.uz`](https://huggingface.co/datasets/wikimedia/wikipedia) | O'zbekcha Vikipediya | ko'p bot-stub maqolalar → < 500 belgi tashlanadi |
| [HuggingFaceFW/fineweb-2 `uzn_Latn`](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2) | Tozalangan o'zbekcha veb matn (~2 GB parquet) | hajm cheklovi bilan oqim (streaming) |
| [OpenCoder-LLM/opc-annealing-corpus](https://huggingface.co/datasets/OpenCoder-LLM/opc-annealing-corpus) | algoritmik kod + sintetik snippetlar (ko'p tilli) | faqat `lang == 'python'`; MBPP'ga o'xshash masalalar bor → decontamination |
| [codeparrot/codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) | GitHub Python fayllari (~50 GB) | turli litsenziyalar → faqat permissive (MIT/Apache/BSD/ISC/CC0) |

Eslatma: `nampdn-ai/tiny-codes` gated, The Stack v2 / StarCoderData esa ruxsat talab qiladi — shuning uchun ishlatilmadi. Ma'lumot yo'q bo'lsa trening **boshlanmaydi**. Litsenziyalarni o'zingiz ham tekshiring.

In [ ]:
# 4.1 Data yordamchilari: o'qish, tozalash, standartlash, decontamination
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '120')
os.environ.setdefault('HF_DATASETS_DISABLE_PROGRESS_BARS', '1')

_SECRET = re.compile(r'-----BEGIN (?:RSA |OPENSSH |EC |DSA )?PRIVATE KEY-----|ghp_[A-Za-z0-9]{30,}|github_pat_[A-Za-z0-9_]{40,}'
                     r'|sk-[A-Za-z0-9_-]{32,}|AKIA[0-9A-Z]{16}|hf_[A-Za-z0-9]{30,}|xox[baprs]-[A-Za-z0-9-]{10,}')
_EMAIL = re.compile(r'(?<![\w.+-])[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}(?!\w)')
_PHONE = re.compile(r'(?<!\d)\+?998[\s-]?\(?\d{2}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2}(?!\d)')
_CODE_RE = re.compile(r'```|^\s*(?:def |class |import |from \S+ import |return\b|for .+:\s*$|while .+:\s*$|if .+:\s*$'
                      r'|print\(|SELECT\b|function\b|#include|public static|const |let |console\.log)', re.M)

def clean_text(v, max_chars=40000, drop_long=False):
    """Maxfiy kalitli hujjatni tashlaydi, email/telefonni maskalaydi, maxsus tokenlarni olib tashlaydi."""
    if not isinstance(v, str):
        return ''
    v = v.replace('\x00', '').strip()
    if not v or _SECRET.search(v):
        return ''
    if len(v) > max_chars:
        if drop_long:
            return ''   # kodni o'rtasidan kesish sintaksisni buzadi
        v = v[:max_chars]
    if '@' in v:
        v = _EMAIL.sub('[EMAIL]', v)
    if '998' in v:
        v = _PHONE.sub('[PHONE]', v)
    return mc.strip_special(v)

def looks_like_code(s):
    return bool(s) and len(_CODE_RE.findall(s[:4000])) >= 2

def stable_hash(x):
    return int(hashlib.md5(str(x).encode('utf-8', 'replace')).hexdigest()[:12], 16)

def parse_tests(tc):
    if isinstance(tc, str):
        try:
            tc = json.loads(tc)
        except Exception:
            tc = [tc]
    if not isinstance(tc, (list, tuple)):
        return []
    return [t.strip() for t in tc if isinstance(t, str) and t.strip().startswith('assert')]

def normalize_turns(turns):
    """user/assistant navbatma-navbat; boshida assistant va oxirida user bo'lmasin."""
    out = []
    for role, text in turns:
        if out and out[-1][0] == role:
            out[-1][1] += '\n\n' + text
        else:
            out.append([role, text])
    while out and out[0][0] != 'user':
        out.pop(0)
    while out and out[-1][0] != 'assistant':
        out.pop()
    return out

# --- O'zbekcha kod so'rov shablonlari (inglizcha ko'rsatma tarjima qilinmaydi, faqat o'raladi) ---
OCI_TEMPLATES = [
    "Quyidagi dasturlash vazifasi uchun ishlaydigan Python funksiyasini yozing.\n\n{t}",
    "Python'da yeching:\n\n{t}",
    "Vazifa:\n{t}\n\nFaqat Python kodini qaytaring.",
    "{t}\n\nYechimni Python'da yozing.",
    "Menga shu masala uchun Python kodi kerak:\n\n{t}",
    "{t}",
]

def oci_prompt(instr, ctx='', variant=0):
    t = instr.strip()
    if ctx and ctx.strip():
        t += "\n\nBoshlang'ich kod:\n```python\n" + ctx.strip() + "\n```"
    return OCI_TEMPLATES[variant % len(OCI_TEMPLATES)].format(t=t)

def render_oci_pt(instr, code, tests):
    """Pretrain uchun 'darslik' ko'rinishidagi fayl: vazifa docstringda, keyin kod va testlar."""
    s = '"""\n' + instr.replace('"""', "'''") + '\n"""\n\n' + code.strip() + '\n'
    if tests:
        s += '\n\n# Testlar\n' + '\n'.join(tests[:5]) + '\n'
    return s

# --- Decontamination: HumanEval/MBPP bilan 10-gram ustma-ust tushish ---
class Decontaminator:
    def __init__(self, n=10):
        self.n, self.grams, self.names = n, set(), set()
        self.name_re = None

    def _grams(self, text):
        w = re.findall(r'\w+', text.lower())
        return {' '.join(w[i:i + self.n]) for i in range(len(w) - self.n + 1)}

    def add(self, text, name=None):
        self.grams |= self._grams(text)
        if name and len(name) >= 6:
            self.names.add(name)

    def finalize(self):
        if self.names:
            self.name_re = re.compile(r'\b(?:' + '|'.join(sorted(map(re.escape, self.names), key=len, reverse=True)) + r')\b')
        return self

    def dirty(self, text):
        if not self.grams or (self.name_re is not None and not self.name_re.search(text)):
            return False
        return not self._grams(text).isdisjoint(self.grams)

# --- Manbalarni o'qish ---
def hf_rows(repo, name=None, split='train', columns=None, limit=None):
    from datasets import load_dataset
    kw = dict(split=split, streaming=True)
    if name:
        kw['name'] = name
    ds = None
    if columns:
        try:
            ds = load_dataset(repo, columns=columns, **kw)
        except (TypeError, ValueError):
            ds = None
    if ds is None:
        ds = load_dataset(repo, **kw)
    it = iter(ds)
    return islice(it, limit) if limit else it

def stream_local_file(path, limit):
    path = Path(path); ext = path.suffix.lower(); n = 0
    if ext in ('.jsonl', '.txt'):
        with path.open('r', encoding='utf-8', errors='replace') as f:
            for line in f:
                if not line.strip():
                    continue
                if ext == '.txt':
                    row = {'text': line.rstrip('\n')}
                else:
                    try:
                        row = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                if isinstance(row, dict):
                    yield row; n += 1
                if n >= limit:
                    return
    elif ext == '.json':
        obj = json.loads(path.read_text('utf-8', errors='replace'))
        if isinstance(obj, dict):
            obj = obj.get('data', obj.get('train', [obj]))
        for row in obj:
            if isinstance(row, str):
                row = {'text': row}
            if isinstance(row, dict):
                yield row; n += 1
            if n >= limit:
                return
    elif ext == '.csv':
        import csv
        csv.field_size_limit(10**8)
        with path.open('r', encoding='utf-8', errors='replace', newline='') as f:
            for row in csv.DictReader(f):
                yield row; n += 1
                if n >= limit:
                    return
    elif ext == '.parquet':
        import pyarrow.parquet as pq
        for batch in pq.ParquetFile(path).iter_batches(batch_size=512):
            for row in batch.to_pylist():
                yield row; n += 1
                if n >= limit:
                    return

def standardize_generic(row, label):
    """Har xil lokal formatlarni yagona yozuvga keltiradi: messages / instruction+javob / OCI uslubi / text."""
    if not isinstance(row, dict):
        return None
    msgs = row.get('messages') or row.get('conversations')
    if isinstance(msgs, list) and msgs:
        turns = []
        for m in msgs:
            if not isinstance(m, dict):
                continue
            role = {'human': 'user', 'gpt': 'assistant'}.get(m.get('role') or m.get('from'), m.get('role') or m.get('from'))
            txt = clean_text(m.get('content') or m.get('value'), 16000)
            if role in ('user', 'assistant') and txt:
                turns.append([role, txt])
        turns = normalize_turns(turns)
        if turns:
            code = any(looks_like_code(t) for r, t in turns if r == 'assistant')
            return dict(pt='', chat=turns, chat_pt=True, domain='code' if code else 'uz', code_sft=code, src=label)
    inst = clean_text(row.get('instruction') or row.get('INSTRUCTION') or row.get('prompt') or row.get('question') or '', 12000)
    resp = clean_text(row.get('response') or row.get('RESPONSE') or row.get('output') or row.get('completion')
                      or row.get('answer') or row.get('code') or '', 20000, drop_long=True)
    inp = clean_text(row.get('input') or '', 8000)
    if inst and resp:
        if inp:
            inst = inst + '\n\n' + inp
        code = looks_like_code(resp)
        pt = ''
        if row.get('code') and code:
            pt = render_oci_pt(inst, resp, parse_tests(row.get('testcase') or row.get('test_list')))
        return dict(pt=pt, chat=[['user', inst], ['assistant', resp]], chat_pt=not pt,
                    domain='code' if code else 'uz', code_sft=code, src=label)
    text = clean_text(row.get('text') or row.get('content') or row.get('code') or '', 40000)
    if len(text) >= 25:
        return dict(pt=text, chat=None, domain='code' if looks_like_code(text) else 'uz', src=label)
    return None

def local_files():
    if LOCAL_PATHS:
        return [Path(p) for p in LOCAL_PATHS]
    base = Path('/kaggle/input')
    if not base.exists():
        return []
    ext = {'.jsonl', '.json', '.txt', '.csv', '.parquet'}
    return sorted(p for p in base.rglob('*') if p.is_file() and p.suffix.lower() in ext
                  and 'moonlight_30h' not in p.parts and p.stat().st_size > 0)

# Qo'lda yozilgan va testlar bilan tekshirilgan tabiiy o'zbekcha kod so'rovlari (36 ta).
# Asosiy dataset o'rnini bosmaydi; o'zbekcha savol -> kod uslubi uchun "langar".
CURATED_UZ_CODE = [
    ("Ikkita sonni qo'shib natijani qaytaradigan Python funksiyasini yozing.",
     'def qosh(a, b):\n    return a + b'),
    ("Python'da son juft yoki toq ekanligini tekshiruvchi funksiya yozing.",
     'def juftmi(n):\n    return n % 2 == 0'),
    ('Faktorialni rekursiyasiz hisoblaydigan Python funksiyasi kerak. Manfiy son berilsa xato chiqarsin.',
     'def faktorial(n):\n    if n < 0:\n        raise ValueError("Manfiy son uchun faktorial aniqlanmagan")\n    natija = 1\n    for i in range(2, n + 1):\n        natija *= i\n    return natija'),
    ('Matnni teskari tartibda qaytaruvchi funksiya yozing.',
     'def teskari(matn):\n    return matn[::-1]'),
    ("Ro'yxatdagi eng katta sonni topadigan funksiya yozing. Ro'yxat bo'sh bo'lsa xato bersin.",
     'def eng_katta(sonlar):\n    if not sonlar:\n        raise ValueError("Ro\'yxat bo\'sh")\n    eng = sonlar[0]\n    for son in sonlar[1:]:\n        if son > eng:\n            eng = son\n    return eng'),
    ('Berilgan son tub son ekanligini aniqlovchi Python funksiyasini yozing.',
     'def tubmi(n):\n    if n < 2:\n        return False\n    i = 2\n    while i * i <= n:\n        if n % i == 0:\n            return False\n        i += 1\n    return True'),
    ("Satr palindrom ekanligini tekshiring: katta-kichik harflar, bo'shliq va tinish belgilarini hisobga olmang.",
     'def palindrommi(matn):\n    toza = [ch.lower() for ch in matn if ch.isalnum()]\n    return toza == toza[::-1]'),
    ("Ro'yxatdan takrorlanuvchi elementlarni tartibini saqlagan holda olib tashlang.",
     'def noyob(elementlar):\n    korilgan = set()\n    natija = []\n    for x in elementlar:\n        if x not in korilgan:\n            korilgan.add(x)\n            natija.append(x)\n    return natija'),
    ('Fibonachchi ketma-ketligining n-hadini qaytaruvchi funksiya yozing (F(0)=0, F(1)=1).',
     'def fibonachchi(n):\n    if n < 0:\n        raise ValueError("n manfiy bo\'lmasligi kerak")\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a'),
    ("Ikki sonning eng katta umumiy bo'luvchisini (EKUB) Evklid algoritmi bilan hisoblang.",
     'def ekub(a, b):\n    a, b = abs(a), abs(b)\n    while b:\n        a, b = b, a % b\n    return a'),
    ('Ikki sonning eng kichik umumiy karralisini (EKUK) topuvchi funksiya yozing.',
     'from math import gcd\n\ndef ekuk(a, b):\n    if a == 0 or b == 0:\n        return 0\n    return abs(a * b) // gcd(a, b)'),
    ('Matndagi unli harflar sonini hisoblovchi funksiya yozing.',
     'def unlilar_soni(matn):\n    unlilar = "aeiouAEIOU"\n    return sum(1 for ch in matn if ch in unlilar)'),
    ("Sonlar ro'yxatining o'rtacha qiymatini qaytaring. Bo'sh ro'yxat uchun xato bering.",
     'def ortacha(sonlar):\n    if not sonlar:\n        raise ValueError("Bo\'sh ro\'yxatning o\'rtachasi yo\'q")\n    return sum(sonlar) / len(sonlar)'),
    ("Matndagi har bir so'z necha marta uchrashini lug'at ko'rinishida qaytaruvchi funksiya yozing.",
     'def soz_chastotasi(matn):\n    natija = {}\n    for soz in matn.lower().split():\n        natija[soz] = natija.get(soz, 0) + 1\n    return natija'),
    ("Ikkita saralangan ro'yxatni bitta saralangan ro'yxatga birlashtiruvchi funksiya yozing.",
     'def birlashtir(a, b):\n    i = j = 0\n    natija = []\n    while i < len(a) and j < len(b):\n        if a[i] <= b[j]:\n            natija.append(a[i])\n            i += 1\n        else:\n            natija.append(b[j])\n            j += 1\n    natija.extend(a[i:])\n    natija.extend(b[j:])\n    return natija'),
    ('Binar qidiruv funksiyasini yozing: element topilsa indeksini, topilmasa -1 qaytarsin.',
     'def binar_qidiruv(sonlar, qidirilgan):\n    chap, ong = 0, len(sonlar) - 1\n    while chap <= ong:\n        orta = (chap + ong) // 2\n        if sonlar[orta] == qidirilgan:\n            return orta\n        if sonlar[orta] < qidirilgan:\n            chap = orta + 1\n        else:\n            ong = orta - 1\n    return -1'),
    ("Ro'yxatni pufakcha (bubble sort) usulida o'sish tartibida saralovchi funksiya yozing.",
     'def pufakcha_saralash(sonlar):\n    a = list(sonlar)\n    n = len(a)\n    for i in range(n):\n        almashdi = False\n        for j in range(n - 1 - i):\n            if a[j] > a[j + 1]:\n                a[j], a[j + 1] = a[j + 1], a[j]\n                almashdi = True\n        if not almashdi:\n            break\n    return a'),
    ("Python'da stek (stack) klassini yozing: push, pop, peek va bo'shligini tekshiruvchi metodlar bilan.",
     'class Stek:\n    def __init__(self):\n        self._elementlar = []\n\n    def push(self, x):\n        self._elementlar.append(x)\n\n    def pop(self):\n        if self.bosh_mi():\n            raise IndexError("Stek bo\'sh")\n        return self._elementlar.pop()\n\n    def peek(self):\n        if self.bosh_mi():\n            raise IndexError("Stek bo\'sh")\n        return self._elementlar[-1]\n\n    def bosh_mi(self):\n        return len(self._elementlar) == 0'),
    ("Talaba klassini yarating: ismi va baholar ro'yxati bo'lsin, o'rtacha bahoni qaytaruvchi metod qo'shing.",
     'class Talaba:\n    def __init__(self, ism, baholar=None):\n        self.ism = ism\n        self.baholar = list(baholar or [])\n\n    def baho_qosh(self, baho):\n        self.baholar.append(baho)\n\n    def ortacha_baho(self):\n        if not self.baholar:\n            return 0.0\n        return sum(self.baholar) / len(self.baholar)'),
    ("Ro'yxatdagi juft sonlarning yig'indisini qaytaruvchi funksiya yozing.",
     'def juftlar_yigindisi(sonlar):\n    return sum(x for x in sonlar if x % 2 == 0)'),
    ("Selsiy darajasini Farengeytga o'giruvchi funksiya yozing.",
     'def selsiydan_farengeytga(c):\n    return c * 9 / 5 + 32'),
    ("Lug'atni qiymatlari bo'yicha kamayish tartibida saralab, yangi lug'at qaytaring.",
     'def qiymat_boyicha_saralash(lugat):\n    return dict(sorted(lugat.items(), key=lambda juft: juft[1], reverse=True))'),
    ("Ichma-ich ro'yxatni bir o'lchamli ro'yxatga aylantiruvchi funksiya yozing.",
     'def tekislash(royxat):\n    natija = []\n    for x in royxat:\n        if isinstance(x, list):\n            natija.extend(tekislash(x))\n        else:\n            natija.append(x)\n    return natija'),
    ("Faylni o'qib, undagi qatorlar sonini qaytaradigan funksiya yozing.",
     'def qatorlar_soni(fayl_yoli):\n    with open(fayl_yoli, encoding="utf-8") as f:\n        return sum(1 for _ in f)'),
    ('Berilgan yil kabisa yili ekanligini aniqlovchi funksiya yozing.',
     'def kabisami(yil):\n    return (yil % 4 == 0 and yil % 100 != 0) or yil % 400 == 0'),
    ("Satrdagi har bir so'zning birinchi harfini katta qilib qaytaring.",
     'def bosh_harflar(matn):\n    return " ".join(soz.capitalize() for soz in matn.split())'),
    ("Ro'yxatni n ta elementdan iborat bo'laklarga ajratuvchi funksiya yozing.",
     'def bolaklarga(royxat, n):\n    if n <= 0:\n        raise ValueError("n musbat bo\'lishi kerak")\n    return [royxat[i:i + n] for i in range(0, len(royxat), n)]'),
    ("Ikki ro'yxatning umumiy elementlarini qaytaring: takrorlanmasin, tartib birinchi ro'yxatdagidek bo'lsin.",
     'def umumiy(a, b):\n    b_set = set(b)\n    natija = []\n    for x in a:\n        if x in b_set and x not in natija:\n            natija.append(x)\n    return natija'),
    ("Musbat butun sonni bin() ishlatmasdan ikkilik ko'rinishdagi satrga o'giring.",
     'def ikkilikka(n):\n    if n == 0:\n        return "0"\n    raqamlar = []\n    while n > 0:\n        raqamlar.append(str(n % 2))\n        n //= 2\n    return "".join(reversed(raqamlar))'),
    ("Satrni butun songa o'giring; o'girib bo'lmasa None qaytaring (try/except ishlating).",
     'def songa_ogir(matn):\n    try:\n        return int(matn)\n    except (ValueError, TypeError):\n        return None'),
    ("Matndagi eng uzun so'zni topuvchi funksiya yozing. Matn bo'sh bo'lsa bo'sh satr qaytarsin.",
     'def eng_uzun_soz(matn):\n    sozlar = matn.split()\n    if not sozlar:\n        return ""\n    return max(sozlar, key=len)'),
    ('Kvadrat tenglama ax^2 + bx + c = 0 ning haqiqiy ildizlarini qaytaruvchi funksiya yozing.',
     'import math\n\ndef kvadrat_ildizlar(a, b, c):\n    if a == 0:\n        raise ValueError("a nolga teng bo\'lmasligi kerak")\n    d = b * b - 4 * a * c\n    if d < 0:\n        return ()\n    if d == 0:\n        return (-b / (2 * a),)\n    q = math.sqrt(d)\n    return ((-b - q) / (2 * a), (-b + q) / (2 * a))'),
    ("Ro'yxat elementlarini k qadam o'ngga aylantiruvchi (rotate) funksiya yozing.",
     'def onga_aylantir(royxat, k):\n    if not royxat:\n        return []\n    k %= len(royxat)\n    return royxat[-k:] + royxat[:-k] if k else list(royxat)'),
    ("Ikki so'z anagramma ekanligini (bir xil harflardan tuzilganini) tekshiring.",
     'def anagrammami(a, b):\n    return sorted(a.replace(" ", "").lower()) == sorted(b.replace(" ", "").lower())'),
    ("Ro'yxatdagi sonlarning kvadratlarini list comprehension yordamida qaytaring.",
     'def kvadratlar(sonlar):\n    return [x * x for x in sonlar]'),
    ('Matndagi katta va kichik harflar sonini alohida hisoblab, ikkalasini qaytaring.',
     'def harflar_soni(matn):\n    katta = sum(1 for ch in matn if ch.isupper())\n    kichik = sum(1 for ch in matn if ch.islower())\n    return katta, kichik'),
]
for _q, _a in CURATED_UZ_CODE:
    ast.parse(_a)
print('Data yordamchilari tayyor | qo\'lda yozilgan o\'zbekcha kod misollari:', len(CURATED_UZ_CODE))

In [ ]:
# 4.2 Manbalarni yig'ish -> RAW_DIR/*.jsonl (yagona format). Tayyor `processed/` bo'lsa butunlay o'tkazib yuboriladi.
import socket

class RawSink:
    """Dedup (8-baytli hash), split, belgilar hisobi; har manba alohida jsonl ga yoziladi."""
    def __init__(self, raw_dir):
        self.dir = Path(raw_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.fh, self.seen = {}, set()
        self.chars, self.val_chars, self.n, self.rej = Counter(), Counter(), Counter(), Counter()

    def add(self, rec):
        src = rec['src']
        body = rec.get('pt') or ''
        if rec.get('chat'):
            body += '\x1e' + json.dumps(rec['chat'], ensure_ascii=False)
        h = int.from_bytes(hashlib.blake2b(body.encode('utf-8', 'replace'), digest_size=8).digest(), 'little')
        if h in self.seen:
            self.rej[src + ':dup'] += 1
            return False
        self.seen.add(h)
        size = len(rec.get('pt') or '') + sum(len(t) for _, t in rec.get('chat') or [])
        split = rec.get('split')
        if split is None:
            split = 'val' if stable_hash(rec.get('key', h)) % 10000 < VAL_PCT * 100 else 'train'
            if split == 'val' and self.val_chars[src] >= VAL_MAX_CHARS_PER_SOURCE:
                split = 'train'
        rec['split'] = split
        rec.pop('key', None)
        if split == 'val':
            self.val_chars[src] += size
        self.chars[src] += size
        self.n[(src, split)] += 1
        if src not in self.fh:
            self.fh[src] = (self.dir / f'{re.sub(r"[^A-Za-z0-9_.-]", "_", src)}.jsonl').open('w', encoding='utf-8')
        self.fh[src].write(json.dumps(rec, ensure_ascii=False) + '\n')
        return True

    def close(self):
        for f in self.fh.values():
            f.close()

def dns_ok(host='huggingface.co'):
    try:
        socket.getaddrinfo(host, 443)
        return True
    except OSError as e:
        print('HF DNS ishlamaydi:', repr(e), '-> Settings -> Internet -> On'); return False

def run_source(sink, label, gen, budget=None, every=50_000):
    """gen dan yozuvlarni oladi, budjet (belgilar) to'lguncha. Xatoda yig'ilgan qism saqlanadi."""
    t0, k = time.time(), 0
    try:
        for rec in gen:
            if rec is None:
                sink.rej[label] += 1
                continue
            rec.setdefault('src', label)
            sink.add(rec); k += 1
            if k % every == 0:
                print(f'   {label}: {k:,} yozuv, {sink.chars[label] / 1e6:,.0f}M belgi, {time.time() - t0:.0f}s', flush=True)
            if budget and sink.chars[label] >= budget:
                break
    except Exception as e:  # noqa: BLE001
        print(f'⚠️ {label} uzildi: {repr(e)[:300]} (yig\'ilgan qism saqlandi)')
    print(f'✔ {label}: {sink.n[(label, "train")]:,} train / {sink.n[(label, "val")]:,} val, '
          f'{sink.chars[label] / 1e6:,.1f}M belgi, rad: {sink.rej[label] + sink.rej[label + ":dup"]:,}, {time.time() - t0:.0f}s', flush=True)

# ---------------- manba generatorlari ----------------
def gen_oci(decon, eval_f):
    group_n, sft_n, eval_groups = Counter(), Counter(), set()
    cols = ['id', 'instruction', 'context', 'code', 'testcase', 'entry_point', 'problem_group_id']
    for r in hf_rows('JetBrains-Research/OpenCodeInstruct-Clean', columns=cols, limit=SMOKE_ROWS and SMOKE_ROWS * 2):
        code = clean_text(r.get('code'), 20000, drop_long=True)
        instr = clean_text(r.get('instruction'), 8000, drop_long=True)
        if not code or not instr:
            yield None; continue
        try:
            ast.parse(code)
        except (SyntaxError, ValueError):
            yield None; continue
        g = r.get('problem_group_id') or r.get('id') or code[:200]
        bucket = stable_hash(g) % 100000
        tests = parse_tests(r.get('testcase'))
        if bucket < OCI_EVAL_FRAC * 100000:          # eval guruhlari treningga UMUMAN kirmaydi
            if g not in eval_groups and tests and len(eval_groups) < OCI_EVAL_PROBLEMS:
                eval_groups.add(g)
                eval_f.write(json.dumps({'instruction': instr, 'context': clean_text(r.get('context'), 4000),
                                         'entry_point': r.get('entry_point'), 'tests': tests, 'reference': code},
                                        ensure_ascii=False) + '\n')
            continue
        if group_n[g] >= OCI_PER_PROBLEM_CAP:
            yield None; continue
        group_n[g] += 1
        if decon is not None and decon.dirty(instr + '\n' + code):
            yield None; continue
        chat = None
        if sft_n[g] < OCI_SFT_PER_PROBLEM:
            sft_n[g] += 1
            v = stable_hash(r.get('id') or code)
            chat = [['user', oci_prompt(instr, clean_text(r.get('context'), 4000), v)], ['assistant', code]]
        yield dict(pt=render_oci_pt(instr, code, tests), chat=chat, chat_pt=False, domain='code', code_sft=True,
                   split='val' if bucket < (OCI_EVAL_FRAC + OCI_VAL_FRAC) * 100000 else 'train')
    print(f'   OCI: {len(group_n):,} masala guruhi, {len(eval_groups)} ta eval masala ajratildi')

UZ_CODE_TOPICS = ('code', 'coding', 'program', 'python', 'javascript', 'sql', 'algorithm', 'dastur')

def gen_uz_lm():
    for hf_split, split in (('train', 'train'), ('validation', 'val')):
        for r in hf_rows('ulugbek2013/uzbek-lang-lm', split=hf_split, limit=SMOKE_ROWS):
            turns = []
            for m in r.get('messages') or []:
                txt = clean_text(m.get('content'), 16000)
                if m.get('role') in ('user', 'assistant') and txt:   # system-promptlar tashlanadi
                    turns.append([m['role'], txt])
            turns = normalize_turns(turns)
            if not turns:
                ins, out = clean_text(r.get('instruction'), 16000), clean_text(r.get('output'), 16000)
                if ins and out:
                    turns = [['user', ins], ['assistant', out]]
            if turns:
                topic = str(r.get('topic') or '').lower()
                code = any(w in topic for w in UZ_CODE_TOPICS) or any(looks_like_code(t) for ro, t in turns if ro == 'assistant')
                yield dict(pt='', chat=turns, chat_pt=True, domain='uz', code_sft=code, split=split)
            else:
                txt = clean_text(r.get('text'), 40000)
                yield dict(pt=txt, chat=None, domain='uz', split=split) if len(txt) >= 50 else None

def gen_wiki():
    for r in hf_rows('wikimedia/wikipedia', name='20231101.uz'):
        txt = clean_text(r.get('text'), 60000)
        if len(txt) < 500:
            yield None; continue
        yield dict(pt=f"{r.get('title', '').strip()}\n\n{txt}", chat=None, domain='uz', key=r.get('id'))

def gen_fw2():
    for r in hf_rows('HuggingFaceFW/fineweb-2', name='uzn_Latn', columns=['id', 'text']):
        txt = clean_text(r.get('text'), 60000)
        yield dict(pt=txt, chat=None, domain='uz', key=r.get('id')) if len(txt) >= 200 else None

def gen_opc(config, decon):
    for r in hf_rows('OpenCoder-LLM/opc-annealing-corpus', name=config):
        if (r.get('lang') or r.get('program_lang') or '').lower() != 'python':
            continue
        txt = clean_text(r.get('text'), 30000, drop_long=True)
        if len(txt) < 100 or (decon is not None and decon.dirty(txt)):
            yield None; continue
        yield dict(pt=txt, chat=None, domain='code')

PERMISSIVE = {'mit', 'apache-2.0', 'bsd-2-clause', 'bsd-3-clause', 'isc', 'cc0-1.0', 'unlicense'}

def gen_codeparrot():
    for r in hf_rows('codeparrot/codeparrot-clean'):
        if str(r.get('license', '')).lower() not in PERMISSIVE or str(r.get('autogenerated', 'false')).lower() == 'true':
            yield None; continue
        txt = clean_text(r.get('content'), 30000, drop_long=True)
        if len(txt) < 200:
            yield None; continue
        name = os.path.basename(str(r.get('path') or 'main.py'))
        yield dict(pt=f'# fayl: {name}\n{txt}', chat=None, domain='code', key=r.get('hash'))

def gen_local(path, decon):
    for row in stream_local_file(path, MAX_LOCAL_ROWS_PER_FILE):
        rec = standardize_generic(row, 'local:' + path.name)
        if rec and rec['domain'] == 'code' and decon is not None and decon.dirty((rec.get('pt') or '') + json.dumps(rec.get('chat') or '')):
            rec = None
        yield rec

def gen_curated():
    for q, a in CURATED_UZ_CODE:
        yield dict(pt='', chat=[['user', q], ['assistant', a]], chat_pt=True, domain='code', code_sft=True,
                   split='train', repeat=CURATED_REPEAT)

# ---------------- ishga tushirish ----------------
RAW_DONE = RAW / '_raw_done.json'
if (PROC / 'meta.json').exists():
    print('✅ Tayyorlangan dataset bor (processed/meta.json) — yig\'ish o\'tkazib yuboriladi.')
elif RAW_DONE.exists():
    print('✅ Xom data oldinroq yig\'ilgan:', RAW)
elif RUN_MODE == 'eval_only':
    raise RuntimeError('eval_only uchun tayyorlangan processed/ va checkpoint kerak (RESUME_INPUT).')
else:
    if RAW.exists():
        shutil.rmtree(RAW)
    PROC.mkdir(parents=True, exist_ok=True)
    t_all = time.time()
    network = SOURCE_MODE != 'local' and dns_ok()
    if SOURCE_MODE == 'hf' and not network:
        raise RuntimeError("SOURCE_MODE='hf', lekin internet yo'q.")
    sink = RawSink(RAW)

    # Benchmarklar: eval uchun saqlanadi va decontamination uchun ishlatiladi
    decon = Decontaminator() if DECONTAMINATE else None
    for fn in ('eval_humaneval.jsonl', 'eval_mbpp.jsonl'):
        (PROC / fn).unlink(missing_ok=True)
    if network:
        try:
            he = [dict(r) for r in hf_rows('openai/openai_humaneval', split='test')]
            with (PROC / 'eval_humaneval.jsonl').open('w', encoding='utf-8') as f:
                for r in he:
                    f.write(json.dumps(r, ensure_ascii=False) + '\n')
                    if decon is not None:
                        decon.add(r['prompt'] + '\n' + r['canonical_solution'], r.get('entry_point'))
            print('HumanEval:', len(he))
        except Exception as e:  # noqa: BLE001
            print('⚠️ HumanEval yuklanmadi:', repr(e)[:200])
        try:
            mb_all = []
            for sp in ('train', 'test', 'validation', 'prompt'):
                mb_all += [dict(r) for r in hf_rows('google-research-datasets/mbpp', name='full', split=sp)]
            for r in mb_all:
                m = re.search(r'def\s+(\w+)', r.get('code', ''))
                if decon is not None:
                    decon.add(r.get('text', '') + '\n' + r.get('code', '') + '\n' + '\n'.join(r.get('test_list', [])),
                              m.group(1) if m else None)
            try:
                mb_eval = [dict(r) for r in hf_rows('google-research-datasets/mbpp', name='sanitized', split='test')]
            except Exception:
                mb_eval = [r for r in mb_all if 11 <= int(r.get('task_id', 0)) <= 510]
            with (PROC / 'eval_mbpp.jsonl').open('w', encoding='utf-8') as f:
                for r in mb_eval:
                    f.write(json.dumps(r, ensure_ascii=False, default=str) + '\n')
            print('MBPP decontam:', len(mb_all), '| MBPP eval:', len(mb_eval))
        except Exception as e:  # noqa: BLE001
            print('⚠️ MBPP yuklanmadi:', repr(e)[:200])
    if decon is not None:
        decon.finalize()
        print(f'Decontamination: {len(decon.grams):,} n-gram, {len(decon.names)} funksiya nomi')
        if not decon.grams:
            decon = None

    # 1) foydalanuvchining lokal fayllari
    if SOURCE_MODE != 'hf':
        files = local_files()
        print('Lokal fayllar:', [str(p) for p in files[:20]], '...' if len(files) > 20 else '')
        for p in files:
            run_source(sink, 'local:' + p.name, gen_local(p, decon))
    # 2) HF manbalari (muhimlik tartibida)
    if network:
        with (PROC / 'eval_oci.jsonl').open('w', encoding='utf-8') as eval_f:
            if USE_OCI_CODE:
                run_source(sink, 'hf_oci', gen_oci(decon, eval_f))
        if USE_UZ_LM:
            run_source(sink, 'hf_uz_lm', gen_uz_lm())
        if USE_UZ_WIKI:
            run_source(sink, 'hf_uz_wiki', gen_wiki(), UZ_WIKI_MAX_CHARS)
        if USE_FINEWEB2_UZ:
            run_source(sink, 'hf_fw2_uz', gen_fw2(), FW2_UZ_MAX_CHARS)
        if USE_OPC_ALGO:
            run_source(sink, 'hf_opc_algo', gen_opc('algorithmic_corpus', decon), OPC_ALGO_MAX_CHARS)
        if USE_OPC_SNIPPET:
            run_source(sink, 'hf_opc_snippet', gen_opc('synthetic_code_snippet', decon), OPC_SNIPPET_MAX_CHARS)
        if USE_CODEPARROT:
            run_source(sink, 'hf_codeparrot', gen_codeparrot(), CODEPARROT_MAX_CHARS)
    # 3) qo'lda yozilgan o'zbekcha kod misollari — faqat haqiqiy data bo'lsa (demo o'rnini bosmaydi)
    if sum(sink.chars.values()) > 0:
        run_source(sink, 'curated_uz_code', gen_curated())
    sink.close()

    total = sum(sink.chars.values())
    min_chars = 2e5 if PROFILE == 'smoke' else 5e7
    if total < min_chars:
        raise RuntimeError(f'Data juda kam ({total / 1e6:.1f}M belgi). Internetni yoqing yoki Kaggle Add Input orqali data qo\'shing. '
                           'Demo data bilan uzoq trening qilinmaydi.')
    RAW_DONE.write_text(json.dumps({'chars': dict(sink.chars), 'counts': {f'{k[0]}|{k[1]}': v for k, v in sink.n.items()},
                                    'rejected': dict(sink.rej), 'minutes': round((time.time() - t_all) / 60, 1)},
                                   ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\n=== XOM DATA: {total / 1e9:.2f} GB belgi, {(time.time() - t_all) / 60:.1f} daqiqa ===')
    for s, c in sink.chars.most_common():
        print(f'  {s:22s} {c / 1e6:10,.1f}M belgi | train {sink.n[(s, "train")]:>9,} | val {sink.n[(s, "val")]:>7,}')

## 5. Tokenizer + tokenlash

Byte-level BPE (`VOCAB=32768`, Rust `tokenizers`): har qanday UTF-8 matn yo'qotishsiz qaytariladi. Pre-tokenizer o'zbekcha apostrofli so'zlarni (o'zbek, g'alaba, ma'no) bo'lmaydi, kod indentatsiyasini alohida bo'laklaydi. Tokenizer kod (60%) va o'zbekcha (40%) namunada o'qitiladi. Natija `processed/` ga **uint16** ko'rinishida yoziladi (pretrain: domenlar bo'yicha; SFT: token + loss-mask + indeks). Chat shabloni: `<|bos|><|user|>\n{savol}\n<|assistant|>\n{javob}<|eos|>`. Resume'da tokenizer **qayta o'qitilmaydi**.

In [ ]:
# 5. Tokenizer o'qitish va hamma datani tokenlash -> processed/ (uint16)
def raw_files():
    return sorted(p for p in RAW.glob('*.jsonl'))

def iter_raw(path):
    with path.open(encoding='utf-8') as f:
        for line in f:
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue

def rec_texts(r):
    if r.get('pt'):
        yield r['pt']
    for _, t in r.get('chat') or []:
        yield t

def tokenizer_sample():
    """Har manbadan proporsional, lekin o'zbekcha kamida 40% — o'zbek so'zlari yaxshi bo'laklansin."""
    stats = json.loads(RAW_DONE.read_text('utf-8'))['chars']
    files = raw_files()
    dom_of = {}
    for p in files:
        first = next(iter_raw(p), None)
        dom_of[p] = (first or {}).get('domain', 'code')
    tot = {d: sum(stats.get(p.stem, 0) for p in files if dom_of[p] == d) for d in ('code', 'uz')}
    share = {'code': 0.6, 'uz': 0.4} if tot['uz'] and tot['code'] else {'code': 1.0 * bool(tot['code']), 'uz': 1.0 * bool(tot['uz'])}
    for p in files:
        d = dom_of[p]
        quota = TOKENIZER_SAMPLE_CHARS * share[d] * stats.get(p.stem, 0) / max(1, tot[d])
        got = 0
        for r in iter_raw(p):
            if r.get('split') != 'train':
                continue
            for t in rec_texts(r):
                yield t
                got += len(t)
            if got >= quota:
                break

if (PROC / 'meta.json').exists():
    tok = mc.MoonTokenizer.load(PROC / 'tokenizer.json')
    META = json.loads((PROC / 'meta.json').read_text('utf-8'))
    print('✅ Tayyor tokenizer va data yuklandi | vocab', tok.vocab_size, '| prepared_id', META['prepared_id'])
    if META.get('seq') != SEQ:
        print(f"⚠️ Data SEQ={META.get('seq')} uchun tayyorlangan, hozir SEQ={SEQ}.")
else:
    t0 = time.time()
    tok = mc.MoonTokenizer.train(tokenizer_sample(), vocab_size=VOCAB)
    tok.save(PROC / 'tokenizer.json')
    print(f'Tokenizer o\'qitildi: vocab={tok.vocab_size}, {time.time() - t0:.0f}s')
    for s in ("Salom, O'zbekiston! G'alaba, ma'no, ko'ngil 🧠", "def summa(a, b):\n    return a + b\n", 'кирилл \t 😀 2026'):
        assert tok.decode(tok.encode(s)) == s, s
    print("Bo'laklash namunasi:", [tok.tk.id_to_token(i) for i in tok.encode("O'zbekiston g'alabasi uchun funksiya yozing")])

    # --- hammasini tokenlash ---
    t0 = time.time()
    MAX_SFT = SEQ + (2 if MTP_WEIGHT > 0 else 1)
    pt_out = {(d, s): (PROC / f'pt_{d}_{s}.bin').open('wb') for d in ('code', 'uz') for s in ('train', 'val')}
    sft = {s: dict(tok=(PROC / f'sft_{s}_tokens.bin').open('wb'), mask=(PROC / f'sft_{s}_mask.bin').open('wb'),
                   idx=[], pos=0) for s in ('train', 'val')}
    ntok, nchar = Counter(), Counter()
    sft_src, sft_long = Counter(), Counter()

    def write_pt(dom, split, ids):
        a = np.asarray(ids, dtype=np.uint16)
        a.tofile(pt_out[(dom, split)])
        ntok[(dom, split)] += len(a)

    def flush(batch):
        pts = [r for r in batch if r.get('pt')]
        if pts:
            for r, ids in zip(pts, tok.encode_batch([r['pt'] for r in pts])):
                write_pt(r['domain'], r['split'], [tok.bos_id] + ids + [tok.eos_id])
                nchar[r['domain']] += len(r['pt'])
        for r in batch:
            if not r.get('chat'):
                continue
            ids, mask = mc.chat_ids(tok, r['chat'])
            if r.get('chat_pt'):
                write_pt(r['domain'], r['split'], ids)
                nchar[r['domain']] += sum(len(t) for _, t in r['chat'])
            if len(ids) > MAX_SFT:
                sft_long[r['src']] += 1
                continue
            S = sft[r['split']]
            np.asarray(ids, dtype=np.uint16).tofile(S['tok'])
            np.asarray(mask, dtype=np.uint8).tofile(S['mask'])
            for _ in range(int(r.get('repeat', 1))):
                S['idx'].append((S['pos'], len(ids), int(bool(r.get('code_sft')))))
            S['pos'] += len(ids)
            sft_src[(r['src'], r['split'])] += 1

    for p in raw_files():
        t1, batch = time.time(), []
        for r in iter_raw(p):
            if r.get('split') not in ('train', 'val'):
                continue
            batch.append(r)
            if len(batch) >= 2048:
                flush(batch); batch = []
        if batch:
            flush(batch)
        print(f'  tokenlandi: {p.stem:22s} {time.time() - t1:6.0f}s', flush=True)
    for f in pt_out.values():
        f.close()
    for s, S in sft.items():
        S['tok'].close(); S['mask'].close()
        np.save(PROC / f'sft_{s}_index.npy', np.asarray(S['idx'], dtype=np.int64).reshape(-1, 3))

    raw_stats = json.loads(RAW_DONE.read_text('utf-8'))
    META = {
        'format': 'moonlight-v3', 'vocab_size': tok.vocab_size, 'seq': SEQ,
        'pad_id': tok.pad_id, 'bos_id': tok.bos_id, 'eos_id': tok.eos_id, 'user_id': tok.user_id, 'assistant_id': tok.assistant_id,
        'pretrain_tokens': {f'{d}_{s}': ntok[(d, s)] for d in ('code', 'uz') for s in ('train', 'val')},
        'chars_per_token': {d: round(nchar[d] / max(1, ntok[(d, 'train')] + ntok[(d, 'val')]), 3) for d in ('code', 'uz')},
        'sft_counts': {f'{a}|{b}': v for (a, b), v in sft_src.items()},
        'sft_too_long': dict(sft_long),
        'sft_train': len(sft['train']['idx']), 'sft_val': len(sft['val']['idx']),
        'sft_code_train': sum(1 for x in sft['train']['idx'] if x[2]),
        'raw': raw_stats, 'seed': SEED, 'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    META['prepared_id'] = hashlib.sha256((PROC / 'tokenizer.json').read_bytes()
                                         + json.dumps(META, sort_keys=True).encode()).hexdigest()[:16]
    (PROC / 'meta.json').write_text(json.dumps(META, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Tokenlash tugadi: {(time.time() - t0) / 60:.1f} daqiqa')
    if not KEEP_RAW:
        shutil.rmtree(RAW, ignore_errors=True)

# --- Data auditi: treningdan oldin albatta ko'ring ---
PT = META['pretrain_tokens']
TRAIN_TOKENS = PT['code_train'] + PT['uz_train']
CODE_SFT_COUNT = META['sft_code_train']
print('\n=== DATA AUDIT ===')
print(f"Pretrain tokenlar: kod {PT['code_train'] / 1e6:,.1f}M + o'zbekcha {PT['uz_train'] / 1e6:,.1f}M = {TRAIN_TOKENS / 1e6:,.1f}M "
      f"(val: {PT['code_val'] / 1e6:.1f}M / {PT['uz_val'] / 1e6:.1f}M)")
print('Belgi/token (yuqori = samaraliroq):', META['chars_per_token'])
print(f"SFT: {META['sft_train']:,} train (kod: {CODE_SFT_COUNT:,}), {META['sft_val']:,} val | SEQ ga sig'magan: {META['sft_too_long']}")
for k, v in sorted(META['sft_counts'].items()):
    print(f'   SFT {k}: {v:,}')
if TRAIN_TOKENS:
    est_tps = 25_000 if PRESET == 'base' else 40_000
    print(f"Taxminiy: {PRETRAIN_HOURS:.0f} soat x ~{est_tps:,} tok/s ≈ {PRETRAIN_HOURS * 3600 * est_tps / 1e9:.1f}B token "
          f"≈ {PRETRAIN_HOURS * 3600 * est_tps / TRAIN_TOKENS:.1f} epoch (4 dan ko'p bo'lsa data qo'shish foydali)")
if CODE_SFT_COUNT < 1000:
    print('⚠️ Kod SFT misollari juda kam. Kod instruction+javob dataset qo\'shing!')

## 6. DDP worker (`train_worker.py`)

`torchrun` har bir T4 uchun alohida jarayon ochadi. Asosiy xususiyatlar:
- **Pretrain**: domenlar aralash nisbati (`CODE_MIX`), WSD LR jadvali (vaqtga bog'liq — seanslar orasida to'g'ri davom etadi).
- **SFT**: chat misollari packing bilan, loss faqat javob tokenlarida; val loss yaxshilanmasa erta to'xtaydi.
- fp16 + GradScaler: overflow qadamlari avtomatik o'tkazib yuboriladi. OOM bo'lsa micro-batch avtomatik kamayadi (global batch saqlanadi).
- Har `SAVE_EVERY_MIN` da atomik checkpoint, `best.pt` val bo'yicha, `manifest.json` progress bilan. Ikkala GPU ham baholashda qatnashadi.

In [ ]:
%%writefile {ROOT}/train_worker.py
# AUTOGENERATED by the MoonLight notebook (%%writefile). Alohida tahrirlamang.
"""MoonLight DDP worker: torchrun bilan har GPU uchun bitta jarayon.
Bosqichlar: pretrain (WSD LR) -> SFT (cosine LR). Vaqt byudjeti seanslar orasida checkpoint orqali davom etadi."""
import argparse
import contextlib
import datetime
import json
import math
import os
import random
import signal
import sys
import time
from pathlib import Path

import warnings

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.insert(0, str(Path(__file__).resolve().parent))
import moonlight_core as mc  # noqa: E402

STOP = False


def _on_signal(signum, frame):
    global STOP
    STOP = True


warnings.filterwarnings("ignore", message=".*broadcast_buffers.*")
for _s in (signal.SIGTERM, signal.SIGINT):
    signal.signal(_s, _on_signal)


def utcnow():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()


# --------------------------------------------------------------------------- data
class PretrainData:
    """Domenlar bo'yicha (code / uz) uint16 token oqimlari; tasodifiy oynalar, aralash nisbati bilan."""

    def __init__(self, pdir, seq, code_mix, need2):
        self.seq, self.mix, self.L = seq, code_mix, seq + (2 if need2 else 1)
        self.bins = {}
        for dom in ("code", "uz"):
            for split in ("train", "val"):
                p = Path(pdir) / f"pt_{dom}_{split}.bin"
                if p.exists() and p.stat().st_size // 2 > self.L + 8:
                    self.bins[(dom, split)] = np.memmap(p, dtype=np.uint16, mode="r")
        self.doms = {s: [d for d in ("code", "uz") if (d, s) in self.bins] for s in ("train", "val")}
        if not self.doms["train"] or not self.doms["val"]:
            raise RuntimeError("Pretrain train/val token fayllari yetarli emas (SEQ dan kichik).")

    def _pick_dom(self, split, rng):
        d = self.doms[split]
        if len(d) == 1:
            return d[0]
        return "code" if rng.random() < self.mix else "uz"

    def batch(self, split, B, rng, dom=None):
        rows = []
        for _ in range(B):
            dd = dom or self._pick_dom(split, rng)
            arr = self.bins[(dd, split)]
            s = int(rng.integers(0, len(arr) - self.L))
            rows.append(np.asarray(arr[s:s + self.L], dtype=np.int64))
        w = torch.from_numpy(np.stack(rows))
        x, y = w[:, :self.seq], w[:, 1:self.seq + 1]
        y2 = w[:, 2:self.seq + 2] if self.L == self.seq + 2 else None
        return x, y, y2


class SFTData:
    """Chat misollari; bir nechta misol bitta qatorga joylanadi (packing), loss faqat javob tokenlarida."""

    def __init__(self, pdir, seq, code_ratio, need2, pad_id):
        self.seq, self.ratio, self.L, self.pad = seq, code_ratio, seq + (2 if need2 else 1), pad_id
        self.d = {}
        for split in ("train", "val"):
            pdir = Path(pdir)
            idx = np.load(pdir / f"sft_{split}_index.npy")
            if len(idx) == 0:
                raise RuntimeError(f"SFT {split} bo'sh")
            self.d[split] = dict(
                tok=np.memmap(pdir / f"sft_{split}_tokens.bin", dtype=np.uint16, mode="r"),
                mask=np.memmap(pdir / f"sft_{split}_mask.bin", dtype=np.uint8, mode="r"),
                idx=idx, code=np.nonzero(idx[:, 2] == 1)[0])
        if len(self.d["train"]["code"]) == 0:
            raise RuntimeError("SFT kod namunalari 0 ta. Kod dataset qo'shing.")
        self.pending = None

    def _pick(self, split, rng):
        d = self.d[split]
        if split == "train" and self.pending is not None:
            j, self.pending = self.pending, None
            return j
        if split == "train" and rng.random() < self.ratio:
            return int(d["code"][rng.integers(0, len(d["code"]))])
        return int(rng.integers(0, len(d["idx"])))

    def batch(self, split, B, rng):
        d = self.d[split]
        X = np.full((B, self.L), self.pad, dtype=np.int64)
        M = np.zeros((B, self.L), dtype=np.uint8)
        for i in range(B):
            pos = 0
            while pos < self.L:
                j = self._pick(split, rng)
                s, n, _ = (int(v) for v in d["idx"][j])
                if n > self.L - pos:
                    if pos == 0:
                        n = self.L
                    else:
                        if split == "train":
                            self.pending = j  # keyingi qatorga o'tkaziladi (qisqa misollarga bias bo'lmasin)
                        break
                X[i, pos:pos + n] = d["tok"][s:s + n]
                M[i, pos:pos + n] = d["mask"][s:s + n]
                pos += n
        Xt, Mt = torch.from_numpy(X), torch.from_numpy(M.astype(np.bool_))
        x = Xt[:, :self.seq]
        y = torch.where(Mt[:, 1:self.seq + 1], Xt[:, 1:self.seq + 1], torch.full_like(x, -100))
        y2 = None
        if self.L == self.seq + 2:
            y2 = torch.where(Mt[:, 2:], Xt[:, 2:], torch.full_like(x, -100))
        return x, y, y2


# --------------------------------------------------------------------------- utils
def atomic_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def save_json(path, data):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)


def ddp_setup(cfg):
    rank = int(os.environ.get("RANK", 0))
    local = int(os.environ.get("LOCAL_RANK", 0))
    world = int(os.environ.get("WORLD_SIZE", 1))
    use_cuda = torch.cuda.is_available() and not cfg.get("force_cpu")
    if use_cuda:
        torch.cuda.set_device(local)
    device = torch.device(f"cuda:{local}" if use_cuda else "cpu")
    if world > 1:
        dist.init_process_group("nccl" if use_cuda else "gloo", timeout=datetime.timedelta(minutes=30))
    return rank, local, world, device


def bcast(values, device, world):
    """Rank 0 qarorlarini (float ro'yxati) barcha ranklarga tarqatadi."""
    t = torch.tensor(values, dtype=torch.float64, device=device)
    if world > 1:
        dist.broadcast(t, src=0)
    return t.tolist()


def allreduce_sum(values, device, world):
    t = torch.tensor(values, dtype=torch.float64, device=device)
    if world > 1:
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return t.tolist()


def barrier(world):
    if world > 1:
        dist.barrier()


def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (decay if p.ndim >= 2 else no_decay).append(p)
    return [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]


def lr_factor(step, progress, warmup, min_ratio, kind, decay_frac):
    w = min(1.0, step / max(1, warmup))
    progress = min(max(progress, 0.0), 1.0)
    if kind == "wsd":  # warmup-stable-decay: vaqt byudjeti noma'lum/uzayadigan treninglar uchun barqaror
        start = 1.0 - decay_frac
        if progress <= start:
            f = 1.0
        else:
            q = (progress - start) / max(1e-9, decay_frac)
            f = min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * q))
    else:
        f = min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return w * f


def make_model(cfg, device):
    c = mc.Config(vocab_size=cfg["vocab_size"], max_seq=cfg["seq"], experts=cfg["experts"],
                  top_k=cfg["top_k"], mtp_weight=cfg["mtp_weight"], grad_ckpt=cfg["grad_ckpt"],
                  **mc.PRESETS[cfg["preset"]]).validate()
    return mc.MoonLightFusion(c).to(device)


def probe_micro_batch(model, micro, seq, vocab, device, amp_dtype, need2, world, rank):
    """OOM bo'lsa micro-batchni avtomatik kamaytiradi (global batch ACCUM orqali saqlanadi)."""
    if device.type != "cuda":
        return micro
    nparams = sum(p.numel() for p in model.parameters())
    total = torch.cuda.get_device_properties(device).total_memory
    while micro >= 1:
        ok, peak, x, loss = 1.0, 0, None, None
        try:
            torch.cuda.reset_peak_memory_stats(device)
            x = torch.randint(0, vocab, (micro, seq), device=device)
            with torch.autocast("cuda", dtype=amp_dtype):
                _, loss, _ = model(x, x, x if need2 else None)
            loss.backward()
            peak = torch.cuda.max_memory_allocated(device)
            # AdamW holati (2x fp32) + DDP bucketlari (~1x fp32) hali ajratilmagan bo'lishi mumkin
            if peak + 12 * nparams > 0.93 * total:
                ok = 0.0
        except torch.cuda.OutOfMemoryError:
            ok = 0.0
        finally:
            x = loss = None
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
        t = torch.tensor([ok], device=device)
        if world > 1:
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
        if t.item() > 0:
            if rank == 0:
                print(f"micro-batch {micro} x {seq} OK (peak {peak / 2**30:.1f} GiB / {total / 2**30:.1f} GiB)", flush=True)
            return micro
        if rank == 0:
            print(f"micro-batch {micro} sig'madi -> {micro // 2}", flush=True)
        micro //= 2
    raise RuntimeError("micro-batch=1 ham sig'madi: PRESET yoki SEQ ni kichraytiring")


@torch.no_grad()
def evaluate(model, data, stage, cfg, device, amp_dtype, use_amp, rank, world):
    """Ikkala rank ham baholaydi (yarmidan), natija all_reduce bilan yig'iladi."""
    model.eval()
    res = {}
    n_batches = max(1, cfg["eval_batches"] // world)
    doms = data.doms["val"] if stage == "pretrain" else ["sft"]
    for dom in doms:
        rng = np.random.default_rng([cfg["seed"], 777, rank, {"code": 1, "uz": 2, "sft": 3}[dom]])
        tot = cnt = 0.0
        for _ in range(n_batches):
            if stage == "pretrain":
                x, y, _ = data.batch("val", cfg["micro"], rng, dom=dom)
            else:
                x, y, _ = data.batch("val", cfg["micro"], rng)
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                _, loss, _ = model(x, y)
            n = float((y != -100).sum())
            if math.isfinite(float(loss)):
                tot += float(loss) * n
                cnt += n
        tot, cnt = allreduce_sum([tot, cnt], device, world)
        res[dom] = tot / cnt if cnt else float("inf")
    model.train()
    # Umumiy val: pretrainda domenlar aralash nisbati bilan
    if stage == "pretrain" and len(res) == 2:
        res["all"] = cfg["code_mix"] * res["code"] + (1 - cfg["code_mix"]) * res["uz"]
    else:
        res["all"] = next(iter(res.values()))
    return res


# --------------------------------------------------------------------------- stage
def run_stage(stage, cfg, rank, local, world, device, prepared_id, deadline):
    root = Path(cfg["root"])
    pdir = root / "processed"
    out = root / stage
    out.mkdir(exist_ok=True)
    last, bestf = out / "last.pt", out / "best.pt"
    budget = float(cfg[f"{stage}_hours"]) * 3600
    is_pt = stage == "pretrain"
    need2 = cfg["mtp_weight"] > 0
    if rank == 0:
        print(f"\n=== {stage.upper()} | byudjet {budget / 3600:.2f} soat ===", flush=True)

    ck = None
    model = make_model(cfg, device)
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)  # faqat o'zingiz yaratgan fayl
        if ck.get("prepared_id") != prepared_id:
            raise RuntimeError("Checkpoint boshqa dataset/tokenizer bilan yaratilgan (prepared_id mos emas).")
        model.load_state_dict(ck["model"])
        if ck.get("complete"):
            if rank == 0:
                print(f"{stage}: allaqachon tugagan (step={ck['step']}).", flush=True)
            return "complete"
    elif not is_pt:
        init = root / "pretrain" / "best.pt"
        if not init.exists():
            init = root / "pretrain" / "last.pt"
        ick = torch.load(init, map_location="cpu", weights_only=False)
        if ick.get("prepared_id") != prepared_id:
            raise RuntimeError("Pretrain checkpoint prepared_id mos emas")
        model.load_state_dict(ick["model"])
        if rank == 0:
            print("SFT boshlang'ich vaznlar:", init, flush=True)
        del ick

    st = dict(step=0, elapsed=0.0, best=float("inf"), best_step=0, no_improve=0, tokens=0, skipped=0)
    if ck is not None:
        for k in st:
            st[k] = ck.get("state", {}).get(k, st[k])
    if st["elapsed"] >= budget:
        return "complete"
    if ck is not None and rank == 0:
        print(f"RESUME {stage}: step={st['step']}, {st['elapsed'] / 3600:.2f}/{budget / 3600:.2f} soat, "
              f"tokens={st['tokens'] / 1e9:.3f}B, best_val={st['best']:.4f}", flush=True)

    use_amp = device.type == "cuda"
    amp_dtype = torch.float16
    if use_amp and torch.cuda.get_device_capability(device)[0] >= 8 and torch.cuda.is_bf16_supported():
        amp_dtype = torch.bfloat16
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16, init_scale=2.0 ** 14)
    else:  # eski PyTorch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16, init_scale=2.0 ** 14)
    lr_peak = cfg["lr_pretrain"] if is_pt else cfg["lr_sft"]
    opt = torch.optim.AdamW(param_groups(model, cfg["weight_decay"]), lr=lr_peak, betas=(0.9, 0.95), eps=1e-8)
    if ck is not None:
        opt.load_state_dict(ck["optimizer"])
        if ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
    del ck

    micro0 = cfg["batch_per_gpu"]
    accum0 = cfg["accum"] if is_pt else cfg["accum_sft"]
    micro = probe_micro_batch(model, micro0, cfg["seq"], cfg["vocab_size"], device, amp_dtype, need2, world, rank)
    accum = max(1, accum0 * micro0 // micro)
    cfg = dict(cfg, micro=micro)
    tok_per_step = micro * accum * cfg["seq"] * world
    if rank == 0:
        print(f"AMP={amp_dtype if use_amp else 'fp32'} | params={model.num_params() / 1e6:.1f}M "
              f"(embedding'siz {model.num_params(True) / 1e6:.1f}M) | micro={micro} accum={accum} "
              f"world={world} -> {tok_per_step:,} token/qadam", flush=True)

    net = model
    if cfg.get("compile") and device.type == "cuda":
        try:
            net = torch.compile(model)
        except Exception as e:  # noqa: BLE001
            if rank == 0:
                print("torch.compile o'chirildi:", repr(e)[:200], flush=True)
            net = model
    if world > 1:
        net = DDP(net, device_ids=[local] if use_amp else None, output_device=local if use_amp else None,
                  broadcast_buffers=False, find_unused_parameters=bool(cfg["experts"]),
                  gradient_as_bucket_view=True)

    if is_pt:
        data = PretrainData(pdir, cfg["seq"], cfg["code_mix"], need2)
    else:
        meta = json.loads((pdir / "meta.json").read_text("utf-8"))
        data = SFTData(pdir, cfg["seq"], cfg["code_sft_ratio"], need2, meta["pad_id"])
        if rank == 0:
            print(f"SFT: {len(data.d['train']['idx']):,} train ({len(data.d['train']['code']):,} kod), "
                  f"{len(data.d['val']['idx']):,} val", flush=True)
    rng = np.random.default_rng([cfg["seed"], rank, int(st["step"]), 0 if is_pt else 1])  # resume'da yangi namunalar
    warm = cfg["warmup_steps"] if is_pt else cfg["warmup_steps_sft"]
    kind = "wsd" if is_pt else "cosine"

    t_start = time.monotonic()
    elapsed0 = st["elapsed"]
    last_save = last_eval = last_log = time.monotonic()
    log_tokens = 0
    session_steps = 0
    bad_streak = 0

    def elapsed():
        return elapsed0 + (time.monotonic() - t_start)

    def save_last(complete=False):
        if rank == 0:
            st["elapsed"] = elapsed()
            atomic_save({"model": model.state_dict(), "config": vars(model.cfg).copy(),
                         "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                         "state": dict(st), "complete": complete, "stage": stage,
                         "prepared_id": prepared_id, "saved_utc": utcnow()}, last)
            man = json.loads((root / "manifest.json").read_text("utf-8"))
            man.setdefault("stages", {})[stage] = {
                "step": st["step"], "elapsed_hours": round(st["elapsed"] / 3600, 3),
                "budget_hours": round(budget / 3600, 3), "tokens_seen": st["tokens"],
                "best_val": st["best"], "best_step": st["best_step"], "complete": complete}
            man["last_updated_utc"] = utcnow()
            save_json(root / "manifest.json", man)
            print(f"💾 checkpoint: {stage}/{last.name} step={st['step']} | {st['elapsed'] / 3600:.2f}/{budget / 3600:.2f} soat", flush=True)
        barrier(world)

    def do_eval(tag):
        res = evaluate(model, data, stage, cfg, device, amp_dtype, use_amp, rank, world)
        v = res["all"]
        improved = math.isfinite(v) and v < st["best"] - 1e-4
        if improved:
            st["best"], st["best_step"], st["no_improve"] = v, st["step"], 0
            if rank == 0:
                atomic_save({"model": model.state_dict(), "config": vars(model.cfg).copy(),
                             "prepared_id": prepared_id, "step": st["step"], "val": res, "stage": stage}, bestf)
        else:
            st["no_improve"] += 1
        if rank == 0:
            pretty = " ".join(f"{k}={v:.4f}" for k, v in res.items())
            print(f"{tag}: {stage} step={st['step']} val[{pretty}] ppl={math.exp(min(20, v)):.2f}"
                  f"{' ⭐ best' if improved else ''}", flush=True)
        barrier(world)

    model.train()
    opt.zero_grad(set_to_none=True)
    while True:
        s = st["step"] + 1
        f = lr_factor(s, elapsed() / budget, warm, cfg["min_lr_ratio"], kind, cfg["decay_frac"])
        lr = lr_peak * f
        for g in opt.param_groups:
            g["lr"] = lr
        loss_acc = torch.zeros((), device=device)
        for m in range(accum):
            x, y, y2 = data.batch("train", micro, rng)
            if use_amp:
                x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
                y2 = y2.pin_memory().to(device, non_blocking=True) if y2 is not None else None
            ctx = net.no_sync() if (world > 1 and m < accum - 1) else contextlib.nullcontext()
            with ctx:
                with torch.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                    _, loss, _ = net(x, y, y2)
                scaler.scale(loss / accum).backward()
            loss_acc += loss.detach()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        if scaler.is_enabled():
            prev = scaler.get_scale()
            scaler.step(opt)          # inf/NaN gradientda qadam avtomatik o'tkazib yuboriladi
            scaler.update()
            skipped = scaler.get_scale() < prev
        else:
            skipped = not bool(torch.isfinite(gn))
            if not skipped:
                opt.step()
        opt.zero_grad(set_to_none=True)
        st["step"] = s
        session_steps += 1
        st["tokens"] += tok_per_step
        log_tokens += tok_per_step
        if skipped:
            st["skipped"] += 1
            bad_streak += 1
            if bad_streak >= 30:
                raise FloatingPointError("30 ta ketma-ket NaN/Inf qadam: LR ni kamaytiring yoki oxirgi last.pt dan davom eting.")
        else:
            bad_streak = 0

        now = time.monotonic()
        if rank == 0 and (s == 1 or s % cfg["log_every_steps"] == 0):
            tps = log_tokens / max(1e-6, now - last_log)
            left_h = max(0.0, budget - elapsed()) / 3600
            print(f"{stage} step={s} loss={float(loss_acc) / accum:.4f} lr={lr:.2e} gnorm={float(gn):.2f} "
                  f"tok/s={tps:,.0f} tokens={st['tokens'] / 1e9:.3f}B elapsed={elapsed() / 3600:.2f}h "
                  f"qoldi={left_h:.2f}h skipped={st['skipped']}"
                  + (f" scale={scaler.get_scale():.0f}" if scaler.is_enabled() else ""), flush=True)
            last_log, log_tokens = now, 0

        # Rank 0 qaror qiladi, hamma rank bir xil yo'ldan yuradi (DDP deadlock bo'lmasligi uchun).
        max_steps = cfg["max_steps_per_session"]
        want_eval = (now - last_eval >= cfg["eval_every_min"] * 60) or (max_steps and session_steps >= max_steps)
        want_end = (time.time() >= deadline - 120) or (elapsed() >= budget) or STOP \
            or bool(max_steps and session_steps >= max_steps)
        want_save = now - last_save >= cfg["save_every_min"] * 60
        flags = bcast([float(want_eval), float(want_end), float(want_save)] if rank == 0 else [0.0, 0.0, 0.0],
                      device, world)
        want_eval, want_end, want_save = (bool(v) for v in flags)
        if want_end and not want_eval:
            want_eval = True
        if want_eval:
            do_eval("FINAL VALIDATION" if want_end else "VALIDATION")
            last_eval = time.monotonic()
        patience = cfg["early_stop_patience"] if is_pt else cfg["early_stop_patience_sft"]
        early = bool(patience) and st["no_improve"] >= patience
        if early and rank == 0:
            print(f"Early stopping: {st['no_improve']} marta val yaxshilanmadi.", flush=True)
        complete = (elapsed() >= budget) or early
        if want_end or early or want_save or want_eval:
            save_last(complete=complete)
            last_save = time.monotonic()
        if want_end or early:
            break
    return "complete" if complete else "session_end"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    cfg = json.loads(Path(ap.parse_args().config).read_text("utf-8"))
    rank, local, world, device = ddp_setup(cfg)
    seed = cfg["seed"] + rank
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(cfg["seed"])  # model init barcha ranklarda bir xil
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    root = Path(cfg["root"])
    meta = json.loads((root / "processed" / "meta.json").read_text("utf-8"))
    prepared_id = meta["prepared_id"]
    spec_keys = ["preset", "seq", "vocab_size", "experts", "top_k", "mtp_weight"]
    spec = {k: cfg[k] for k in spec_keys}
    man_path = root / "manifest.json"
    if rank == 0 and not man_path.exists():
        save_json(man_path, {"format": "MoonLight30h-v3", "prepared_id": prepared_id, "model_spec": spec,
                             "stages": {}, "created_utc": utcnow()})
    barrier(world)
    man = json.loads(man_path.read_text("utf-8"))
    if man.get("prepared_id") != prepared_id:
        raise RuntimeError("manifest.json boshqa tayyorlangan dataset uchun. Eski checkpointni aralashtirmang.")
    if man.get("model_spec") != spec:
        raise RuntimeError(f"Model sozlamalari checkpointdan farq qiladi: {man.get('model_spec')} != {spec}. "
                           "PRESET/SEQ/VOCAB/EXPERTS/MTP seanslar orasida o'zgarmasligi kerak.")
    deadline = float(cfg["deadline_unix"])
    if rank == 0:
        left = (deadline - time.time()) / 3600
        print(f"DDP world={world} device={device} prepared_id={prepared_id} | seans uchun qolgan vaqt {left:.2f} soat", flush=True)
    try:
        status = run_stage("pretrain", cfg, rank, local, world, device, prepared_id, deadline)
        if device.type == "cuda":
            torch.cuda.empty_cache()
        if status == "complete" and time.time() < deadline - 15 * 60 and not STOP:
            if cfg["sft_hours"] > 0:
                run_stage("sft", cfg, rank, local, world, device, prepared_id, deadline)
        elif rank == 0:
            print("Seans tugadi. Pretrain tugagach SFT keyingi seansda davom etadi.", flush=True)
    finally:
        if world > 1:
            try:
                dist.barrier()
            finally:
                dist.destroy_process_group()


if __name__ == "__main__":
    main()

## 7. **Treningni boshlash** (T4 ×2 DDP)

Logda `DDP world=2` chiqishi kerak. Har `LOG_EVERY_STEPS` qadamda: loss, LR, tok/s, ko'rilgan tokenlar, qolgan vaqt. Trening seans tugashidan `FINAL_RESERVE_MIN` oldin o'zi to'xtaydi va checkpoint saqlaydi.

In [ ]:
# 7. Trening
TRAIN_RAN = False
if RUN_MODE == 'prepare_only':
    print('✅ prepare_only: data tayyor. Save Version qiling va GPU seansida shu Output\'ni Add Input qiling.')
elif RUN_MODE == 'eval_only':
    print('eval_only: trening o\'tkazib yuboriladi.')
else:
    use_cuda = torch.cuda.is_available() and not FORCE_CPU
    if PROFILE == 'real':
        if not use_cuda:
            raise RuntimeError('GPU topilmadi: Settings -> Accelerator -> GPU T4 x2')
        if torch.cuda.device_count() < 2:
            print(f'⚠️ {torch.cuda.device_count()} ta GPU ko\'rinyapti (T4 x2 tavsiya etiladi) — trening sekinroq bo\'ladi.')
        if CODE_SFT_COUNT < 1000:
            raise RuntimeError('Kod SFT misollari 1000 dan kam — avval kod dataset qo\'shing.')
        if TRAIN_TOKENS < 50_000_000:
            raise RuntimeError(f'Pretrain tokenlar atigi {TRAIN_TOKENS:,}. 30 soat uchun juda kam — data qo\'shing.')
    deadline = NOTEBOOK_T0 + SESSION_HOURS * 3600 - FINAL_RESERVE_MIN * 60
    left_min = (deadline - time.time()) / 60
    if left_min < 20:
        print(f'⚠️ Treningga atigi {left_min:.0f} daqiqa qoldi — bu seansda trening boshlanmaydi.')
    else:
        CONF = {'root': str(ROOT), 'preset': PRESET, 'seq': SEQ, 'vocab_size': tok.vocab_size,
                'experts': EXPERTS, 'top_k': TOP_K_EXPERTS, 'mtp_weight': MTP_WEIGHT, 'grad_ckpt': GRAD_CKPT,
                'batch_per_gpu': BATCH_PER_GPU, 'accum': ACCUM, 'accum_sft': ACCUM_SFT, 'seed': SEED,
                'lr_pretrain': LR_PRETRAIN, 'lr_sft': LR_SFT, 'warmup_steps': WARMUP_STEPS, 'warmup_steps_sft': WARMUP_STEPS_SFT,
                'min_lr_ratio': MIN_LR_RATIO, 'decay_frac': DECAY_FRAC, 'weight_decay': WEIGHT_DECAY, 'grad_clip': GRAD_CLIP,
                'code_mix': CODE_MIX, 'code_sft_ratio': CODE_SFT_RATIO,
                'pretrain_hours': PRETRAIN_HOURS, 'sft_hours': SFT_HOURS, 'deadline_unix': deadline,
                'save_every_min': SAVE_EVERY_MIN, 'eval_every_min': EVAL_EVERY_MIN, 'eval_batches': EVAL_BATCHES,
                'log_every_steps': LOG_EVERY_STEPS, 'max_steps_per_session': MAX_STEPS_PER_SESSION,
                'early_stop_patience': EARLY_STOP_PATIENCE, 'early_stop_patience_sft': EARLY_STOP_PATIENCE_SFT,
                'compile': COMPILE, 'force_cpu': FORCE_CPU}
        configfile = ROOT / 'train_config.json'
        configfile.write_text(json.dumps(CONF, ensure_ascii=False, indent=2), encoding='utf-8')
        nproc = int(NPROC) if NPROC != 'auto' else (max(1, torch.cuda.device_count()) if use_cuda else 1)
        print(f'DDP: {nproc} jarayon | treningga {left_min / 60:.2f} soat | jami reja {PRETRAIN_HOURS}h pretrain + {SFT_HOURS}h SFT')
        cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone', f'--nproc_per_node={nproc}',
               str(ROOT / 'train_worker.py'), '--config', str(configfile)]
        env = os.environ.copy()
        env.update(PYTHONUNBUFFERED='1', OMP_NUM_THREADS='2' if not use_cuda else '1',
                   TORCH_NCCL_ASYNC_ERROR_HANDLING='1', NCCL_IB_DISABLE='1')
        env.setdefault('NCCL_P2P_DISABLE', '1')   # Kaggle T4x2 da NCCL P2P osilib qolishining oldini oladi
        proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end='', flush=True)
            rc = proc.wait()
        except KeyboardInterrupt:
            print('\nTo\'xtatilmoqda — worker checkpoint saqlaydi...')
            import signal as _signal
            proc.send_signal(_signal.SIGINT)
            rc = proc.wait(timeout=300)
        if rc:
            raise RuntimeError(f'DDP trening xato bilan to\'xtadi (code {rc}). Yuqoridagi traceback\'ni ko\'ring; '
                               'oxirgi checkpoint saqlangan — keyingi seansda davom etadi.')
        TRAIN_RAN = True
        print('✅ Trening seansi tugadi.')

## 8. Baholash: chat namunalari + **pass@1** (haqiqiy testlar bilan)

- **OCI-holdout**: treningdan butunlay ajratilgan ~200 masala (o'zbekcha shablon bilan so'raladi), har biri 10 ta test.
- **HumanEval** (164) va **MBPP-sanitized** (257): standart benchmarklar (training datadan 10-gram decontamination qilingan).
- **Base HumanEval**: pretrain checkpointi completion rejimida (SFT ta'sirini solishtirish uchun).

Model yozgan kod **alohida jarayonda**, vaqt (6 s) va xotira (1 GB) cheklovi bilan, fayl o'chirish / tizim buyruqlari / tashqi katalogga yozish o'chirilgan holda bajariladi (human-eval uslubidagi himoya). Bu to'liq izolyatsiya emas — faqat bir martalik Kaggle konteynerida ishlating; o'chirish uchun `RUN_EXEC_EVAL=False`. Greedy dekodlash, natija `eval_results.json` ga qo'shib boriladi.

In [ ]:
# 8. Baholash
EVAL_DEADLINE = NOTEBOOK_T0 + SESSION_HOURS * 3600 - 8 * 60
EVAL_DEV = 'cuda:0' if torch.cuda.is_available() and not FORCE_CPU else 'cpu'
EVAL_DTYPE = torch.float16 if EVAL_DEV.startswith('cuda') else None
STOP_IDS = (tok.eos_id, tok.user_id)

def first_existing(*paths):
    return next((p for p in paths if p.exists()), None)

CKPT_CHAT = first_existing(ROOT / 'sft' / 'best.pt', ROOT / 'sft' / 'last.pt', ROOT / 'pretrain' / 'best.pt', ROOT / 'pretrain' / 'last.pt')
CKPT_BASE = first_existing(ROOT / 'pretrain' / 'best.pt', ROOT / 'pretrain' / 'last.pt')

def chat_prompt(question):
    return mc.chat_ids(tok, [('user', question)], add_generation_prompt=True)[0]

def ask(question, max_new_tokens=320, temperature=0.2, top_p=0.95, model=None):
    """Interaktiv foydalanish: print(ask("Python'da ... funksiya yozing"))"""
    model = model or CHAT_MODEL
    out = model.generate(chat_prompt(question), max_new_tokens, temperature=temperature, top_p=top_p,
                         stop_ids=STOP_IDS, pad_id=tok.pad_id, device=EVAL_DEV)
    return tok.decode(out).strip()

def gen_all(model, prompts, max_new):
    """Uzunlik bo'yicha guruhlab batch generatsiya. Vaqt tugasa qolganlari None."""
    outs = [None] * len(prompts)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    for k in range(0, len(order), EVAL_BATCH):
        if time.time() > EVAL_DEADLINE:
            print('⏱ eval vaqti tugadi — qolgan masalalar baholanmadi'); break
        chunk = [i for i in order[k:k + EVAL_BATCH] if len(prompts[i]) < model.cfg.max_seq - 16]
        for i in order[k:k + EVAL_BATCH]:
            if i not in chunk:
                outs[i] = ''   # kontekstga sig'madi -> muvaffaqiyatsiz deb hisoblanadi
        if chunk:
            gens = model.generate_batch([prompts[i] for i in chunk], max_new, temperature=0.0,
                                        stop_ids=STOP_IDS, pad_id=tok.pad_id, device=EVAL_DEV)
            for i, g in zip(chunk, gens):
                outs[i] = tok.decode(g)
    return outs

def pass_at_1(name, programs, show=2):
    pairs = [(i, p) for i, p in enumerate(programs) if p is not None]
    if not pairs:
        return None
    res = mc.run_many([p for _, p in pairs], workers=4, timeout=6.0)
    ok = sum(r[0] for r in res)
    rate = ok / len(pairs)
    print(f'📊 {name}: pass@1 = {rate * 100:.1f}%  ({ok}/{len(pairs)})')
    fails = [(i, r[1]) for (i, _), r in zip(pairs, res) if not r[0]][:show]
    for i, err in fails:
        print(f'   ✗ #{i}: {err.strip().splitlines()[-1][:160] if err.strip() else "?"}')
    return {'pass@1': round(rate, 4), 'passed': ok, 'n': len(pairs)}

def imports_of(src):
    return '\n'.join(l for l in src.splitlines() if l.startswith(('import ', 'from ')))

def load_jsonl(p, n=None):
    if not p.exists():
        return []
    rows = [json.loads(l) for l in p.read_text('utf-8').splitlines() if l.strip()]
    return rows[:n] if n else rows

CHAT_MODEL = None
if not RUN_EVAL or RUN_MODE == 'prepare_only':
    print('Baholash o\'tkazib yuborildi.')
elif CKPT_CHAT is None:
    print('Checkpoint topilmadi — avval trening qiling.')
else:
    CHAT_MODEL, ck_ = mc.load_checkpoint_model(CKPT_CHAT, EVAL_DEV, EVAL_DTYPE)
    print(f"Model: {CKPT_CHAT.relative_to(ROOT)} | step {ck_.get('step', ck_.get('state', {}).get('step'))} | {EVAL_DEV}")
    del ck_
    print('\n=== CHAT NAMUNALARI ===')
    DEMO = ["Python'da berilgan ro'yxatdagi juft sonlarning yig'indisini qaytaradigan funksiya yozing.",
            "Matndagi har bir so'z necha marta uchrashini hisoblaydigan Python funksiyasini yozing.",
            "Ikkita saralangan ro'yxatni bitta saralangan ro'yxatga birlashtiruvchi funksiya yozing.",
            "Berilgan son tub son ekanligini tekshiruvchi Python funksiyasini yozing.",
            "Python'da navbat (queue) klassini enqueue, dequeue va is_empty metodlari bilan yozing."]
    for q in DEMO[:2 if PROFILE == 'smoke' else None]:
        a = ask(q, temperature=0.0)
        code = mc.extract_code(a)
        try:
            ast.parse(code); synt = '✅ sintaksis OK'
        except SyntaxError as e:
            synt = f'❌ sintaksis xatosi: {e.msg}'
        print(f'\nSO\'ROV: {q}\nMODEL:\n{a[:1200]}\n{synt}')

    results = {'time_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'checkpoint': str(CKPT_CHAT.relative_to(ROOT)),
               'manifest': json.loads((ROOT / 'manifest.json').read_text('utf-8')).get('stages') if (ROOT / 'manifest.json').exists() else None}
    if RUN_EXEC_EVAL:
        print('\n=== PASS@1 ===')
        oci = load_jsonl(PROC / 'eval_oci.jsonl', EVAL_OCI_N)
        if oci:
            gens = gen_all(CHAT_MODEL, [chat_prompt(oci_prompt(p['instruction'], p.get('context', ''), 0)) for p in oci], EVAL_MAX_NEW)
            progs = [None if g is None else mc.extract_code(g) + '\n\n' + '\n'.join(p['tests']) for g, p in zip(gens, oci)]
            results['oci_holdout'] = pass_at_1('OCI-holdout (o\'zbekcha so\'rov)', progs)
        he = load_jsonl(PROC / 'eval_humaneval.jsonl') if EVAL_HUMANEVAL else []
        if he:
            tmpl = "Quyidagi Python funksiyasini to'liq yozing (imzo va docstring berilgan):\n\n```python\n{p}```"
            gens = gen_all(CHAT_MODEL, [chat_prompt(tmpl.format(p=p['prompt'])) for p in he], EVAL_MAX_NEW)
            progs = []
            for g, p in zip(gens, he):
                if g is None:
                    progs.append(None); continue
                code = mc.extract_code(g)
                body = code if f"def {p['entry_point']}" in code else p['prompt'] + code
                progs.append(imports_of(p['prompt']) + '\n' + body + '\n\n' + p['test'] + f"\ncheck({p['entry_point']})\n")
            results['humaneval_chat'] = pass_at_1('HumanEval (chat)', progs)
        mb = load_jsonl(PROC / 'eval_mbpp.jsonl') if EVAL_MBPP else []
        if mb:
            qs = [f"Vazifa: {r.get('prompt') or r.get('text')}\nKod quyidagi testdan o'tishi kerak:\n{r['test_list'][0]}" for r in mb]
            gens = gen_all(CHAT_MODEL, [chat_prompt(q) for q in qs], EVAL_MAX_NEW)
            progs = [None if g is None else '\n'.join(r.get('test_imports') or []) + '\n' + (r.get('test_setup_code') or '')
                     + '\n' + mc.extract_code(g) + '\n\n' + '\n'.join(r['test_list']) for g, r in zip(gens, mb)]
            results['mbpp_chat'] = pass_at_1('MBPP-sanitized (chat)', progs)
        if he and EVAL_BASE_HUMANEVAL and CKPT_BASE is not None and time.time() < EVAL_DEADLINE:
            base, _ = mc.load_checkpoint_model(CKPT_BASE, EVAL_DEV, EVAL_DTYPE)
            gens = gen_all(base, [[tok.bos_id] + tok.encode(p['prompt']) for p in he], EVAL_MAX_NEW)
            progs = [None if g is None else p['prompt'] + mc.truncate_completion(g) + '\n\n' + p['test'] + f"\ncheck({p['entry_point']})\n"
                     for g, p in zip(gens, he)]
            results['humaneval_base_completion'] = pass_at_1('HumanEval (pretrain, completion)', progs)
            del base
    hist_p = ROOT / 'eval_results.json'
    hist = json.loads(hist_p.read_text('utf-8')) if hist_p.exists() else []
    hist.append(results)
    hist_p.write_text(json.dumps(hist, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nNatijalar saqlandi:', hist_p)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. Eksport va progress

`export/` ichida: `model_fp16.pt` (+ `moonlight_core.py`, `tokenizer.json`) va `hf_qwen3/` — standart **transformers** formatida:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained('.../export/hf_qwen3'); model = AutoModelForCausalLM.from_pretrained('.../export/hf_qwen3')
ids = tok.apply_chat_template([{'role': 'user', 'content': "Python'da tub sonni tekshiruvchi funksiya yozing."}], add_generation_prompt=True, return_tensors='pt')
print(tok.decode(model.generate(ids, max_new_tokens=200, do_sample=False)[0][ids.shape[1]:], skip_special_tokens=True))
```
Eksportdan keyin HF model logitlari asl model bilan solishtiriladi.

In [ ]:
# 9. Eksport + progress
EXPORT = ROOT / 'export'
CKPT_EXPORT = first_existing(ROOT / 'sft' / 'best.pt', ROOT / 'sft' / 'last.pt', ROOT / 'pretrain' / 'best.pt', ROOT / 'pretrain' / 'last.pt')
if RUN_MODE != 'prepare_only' and CKPT_EXPORT is not None:
    EXPORT.mkdir(exist_ok=True)
    m_, ck_ = mc.load_checkpoint_model(CKPT_EXPORT, 'cpu')
    torch.save({'model': {k: v.half() for k, v in m_.state_dict().items()}, 'config': vars(m_.cfg).copy(),
                'source_checkpoint': str(CKPT_EXPORT.relative_to(ROOT)), 'prepared_id': ck_.get('prepared_id')},
               EXPORT / 'model_fp16.pt')
    shutil.copy(PROC / 'tokenizer.json', EXPORT / 'tokenizer.json')
    shutil.copy(ROOT / 'moonlight_core.py', EXPORT / 'moonlight_core.py')
    print('✅ export/model_fp16.pt <-', CKPT_EXPORT.relative_to(ROOT))
    if EXPORT_HF and not m_.cfg.experts:
        try:
            from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
            hf_dir = EXPORT / 'hf_qwen3'; hf_dir.mkdir(exist_ok=True)
            (hf_dir / 'config.json').write_text(json.dumps(mc.qwen3_config_dict(m_.cfg, tok), indent=2), encoding='utf-8')
            sd = mc.to_qwen3_state_dict(m_)
            try:
                from safetensors.torch import save_file
                save_file(sd, str(hf_dir / 'model.safetensors'), metadata={'format': 'pt'})
            except ImportError:
                torch.save(sd, hf_dir / 'pytorch_model.bin')
            (hf_dir / 'generation_config.json').write_text(json.dumps(
                {'bos_token_id': tok.bos_id, 'eos_token_id': [tok.eos_id, tok.user_id], 'pad_token_id': tok.pad_id}, indent=2))
            hf_tok = PreTrainedTokenizerFast(tokenizer_file=str(PROC / 'tokenizer.json'), bos_token='<|bos|>', eos_token='<|eos|>',
                                             pad_token='<|pad|>', additional_special_tokens=['<|user|>', '<|assistant|>', '<|system|>'])
            hf_tok.chat_template = mc.CHAT_TEMPLATE
            hf_tok.save_pretrained(str(hf_dir))
            # Tekshiruv: HF model logitlari == asl model logitlari
            try:
                hm = AutoModelForCausalLM.from_pretrained(str(hf_dir), dtype=torch.float32)
            except TypeError:
                hm = AutoModelForCausalLM.from_pretrained(str(hf_dir), torch_dtype=torch.float32)
            hm = hm.float().eval()
            ref = mc.MoonLightFusion(m_.cfg).eval()
            ref.load_state_dict({k: v.half().float() for k, v in m_.state_dict().items()})
            q = "Python'da tub sonni tekshiruvchi funksiya yozing."
            ids_ = torch.tensor([chat_prompt(q)])
            t_ids = hf_tok.apply_chat_template([{'role': 'user', 'content': q}], add_generation_prompt=True, tokenize=True)
            if hasattr(t_ids, 'keys'):
                t_ids = t_ids['input_ids']
            if t_ids and isinstance(t_ids[0], list):
                t_ids = t_ids[0]
            print('HF chat shabloni o\'qitish formatiga mos:', list(t_ids) == ids_[0].tolist())
            with torch.no_grad():
                d = (hm(ids_).logits - ref(ids_)[0]).abs().max().item()
            print(f'✅ HF (Qwen3) eksport: {hf_dir} | logit farqi {d:.2e}' + ('' if d < 1e-2 else '  ⚠️ katta farq!'))
            del hm, ref
        except Exception as e:  # noqa: BLE001
            print('⚠️ HF eksport o\'tkazib yuborildi:', repr(e)[:300])
    del m_, ck_

man_p = ROOT / 'manifest.json'
if man_p.exists():
    man = json.loads(man_p.read_text('utf-8'))
    print('\n=== 30 SOATLIK PROGRESS ===')
    tot = 0.0
    for stage, h in (('pretrain', PRETRAIN_HOURS), ('sft', SFT_HOURS)):
        r = man.get('stages', {}).get(stage, {})
        e = r.get('elapsed_hours', 0.0); tot += e
        print(f"{stage:8s}: {e:6.2f}/{h:.2f} soat | step={r.get('step', 0):,} | tokens={r.get('tokens_seen', 0) / 1e9:.2f}B "
              f"| best_val={r.get('best_val')} | {'✅ tugagan' if r.get('complete') else '…davom etadi'}")
    print(f'Jami: {tot:.2f}/{PRETRAIN_HOURS + SFT_HOURS:.2f} soat')
sizes = sorted(((p.stat().st_size, p) for p in ROOT.rglob('*') if p.is_file()), reverse=True)
print(f'Output hajmi: {sum(s for s, _ in sizes) / 2**30:.2f} GiB (Kaggle limiti 20 GiB) | eng kattalari:',
      ', '.join(f'{p.relative_to(ROOT)} {s / 2**30:.2f}G' for s, p in sizes[:4]))
print('\nKeyingi seans: Save Version -> Output\'ni Add Input qiling -> Save & Run All. Katalog:', ROOT)

## Seansdan keyin (muhim)

1. Seansni **`Save Version → Save & Run All`** bilan ishga tushiring — fon rejimida ishlaydi va tugagach Output avtomatik saqlanadi. Oddiy interaktiv `Run All` + `Quick Save` Output'ni saqlashni kafolatlamaydi.
2. Keyingi seansda: **Add Input → Your Work → shu notebook → oxirgi versiya Output'i**. `RESUME_INPUT` ni bo'sh qoldirsangiz eng ko'p o'qitilgan `moonlight_30h` avtomatik tanlanadi.
3. Logda `DDP world=2`, `RESUME`/`micro-batch ... OK`, `VALIDATION` va `💾 checkpoint` qatorlarini tekshiring.
4. Taxminan 3 ta GPU seansida 26 soat pretrain + 4 soat SFT yakunlanadi. GPU kvotasi tugasa, keyingi haftada davom ettiring — checkpoint yo'qolmaydi.
5. **Sozlamalarni seanslar orasida o'zgartirmang** (`PRESET`, `SEQ`, `VOCAB`, `EXPERTS`, `MTP_WEIGHT`). Data/tokenizer qayta yaratilmaydi — u `processed/` dan olinadi.

### Kuchliroq qilish uchun keyingi qadamlar
- **O'zbekcha kod savol-javoblari** — eng katta bo'shliq. Hozircha kod ko'rsatmalari inglizcha (o'zbekcha shablon bilan o'ralgan) + uzbek-lang-lm dagi dasturlash domeni + 36 ta qo'lda yozilgan misol. O'zingizning tekshirilgan o'zbekcha `instruction`/`response` (yoki `messages`) JSONL faylingizni Add Input qilsangiz, avtomatik qo'shiladi.
- pass@1 natijalarini `eval_results.json` da kuzating: SFT dan keyin HumanEval/OCI-holdout o'sishi kerak.
- Maksimal sifat kerak bo'lsa: tayyor kod modelini (masalan Qwen2.5-Coder-1.5B) LoRA bilan shu o'zbekcha data ustida fine-tune qilish — noldan o'qitishdan ancha kuchli.

**Eslatma:** notebookda `input()` yo'q, `Save & Run All` bloklanmaydi. 30 soat — maksimal vaqt byudjeti; early stopping yoki xato bo'lsa erta tugashi mumkin.